In [1]:
# ============================================================
# ONE-CELL RUN BLOCK (Colab-friendly)
#   1) Prefactor-corrected plateau extractor => Mass measurement module
#   2) SU(2) RG-intertwining constant test (JAX static-arg fix + intrinsic tangent norms)
#   3) CasRG model2: swap Lanczos(full reorth) -> LOBPCG / IRL / shift-invert
#   4) SU(3) commutator seed: surrogate -> true ||[A_mu,A_nu]||^2 + Hessian/safe-radius compare
# ============================================================

import math
import numpy as np
import torch

TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE  = torch.float64
torch.set_default_dtype(TORCH_DTYPE)

# ============================================================
# 1) MASS MEASUREMENT MODULE
# ============================================================

def _to_torch(x, device=TORCH_DEVICE, dtype=TORCH_DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(device=device, dtype=dtype)
    return torch.as_tensor(x, device=device, dtype=dtype)

def tail_envelope_abs(vals: torch.Tensor) -> torch.Tensor:
    """env[r] = max_{s>=r} |vals[s]| (monotone tail envelope)."""
    v = torch.abs(vals)
    v_rev = torch.flip(v, dims=[0])
    env_rev = torch.cummax(v_rev, dim=0)[0]
    return torch.flip(env_rev, dims=[0])

def prefactor_correct(vals: torch.Tensor, power: float) -> torch.Tensor:
    """Multiply by (r+1)^power to remove asymptotic r^{-power} prefactor."""
    r = torch.arange(vals.numel(), device=vals.device, dtype=vals.dtype) + 1.0
    return (r ** power) * vals

def local_slopes_from_env(env: torch.Tensor, eps: float = 1e-300):
    """slopes[r] = -(log env[r+1] - log env[r]) with mids[r] = r+0.5."""
    y = torch.log(torch.clamp(env, min=eps))
    slopes = -(y[1:] - y[:-1])
    mids = torch.arange(0.5, 0.5 + slopes.numel(), device=env.device, dtype=env.dtype)
    return mids, slopes

def robust_plateau(slopes: torch.Tensor,
                   mids: torch.Tensor,
                   rmin: float,
                   rmax: float,
                   win: int = 8):
    """
    Pick the window of length win inside [rmin,rmax] with smallest MAD.
    Returns (m_hat, mad_hat, (r_left, r_right)).
    """
    mask = (mids >= rmin) & (mids <= rmax)
    s = slopes[mask]
    x = mids[mask]
    n = s.numel()
    if n < win:
        raise ValueError(f"Not enough slope points in [{rmin},{rmax}] (have {n}, need >= {win}).")

    best_m = None
    best_mad = None
    best_i = None
    for i in range(0, n - win + 1):
        w = s[i:i+win]
        med = torch.median(w)
        mad = torch.median(torch.abs(w - med))
        if (best_mad is None) or (mad < best_mad):
            best_mad = mad
            best_m = med
            best_i = i

    r_left  = float(x[best_i].item())
    r_right = float(x[best_i + win - 1].item())
    return float(best_m.item()), float(best_mad.item()), (r_left, r_right)

def torus_L1_dist_grid(L: int, d: int, device=TORCH_DEVICE):
    """dist[x] = sum_i min(x_i, L-x_i) on a d-dimensional L-torus."""
    coords = torch.meshgrid(*([torch.arange(L, device=device)] * d), indexing="ij")
    dist = torch.zeros([L] * d, device=device, dtype=torch.int32)
    for ax in range(d):
        x = coords[ax]
        dist += torch.minimum(x, (L - x))
    return dist

def env_by_dist_max(vals: torch.Tensor, dist: torch.Tensor, r_max: int):
    """env[r] = max_{x: dist[x]=r} vals[x]."""
    v = vals.reshape(-1)
    d = dist.reshape(-1).to(torch.int64)
    env = torch.zeros((r_max + 1,), device=vals.device, dtype=vals.dtype)
    return env.scatter_reduce(0, d, v, reduce="amax", include_self=True)

def directional_profile_abs(G: torch.Tensor,
                            direction,
                            L: int,
                            d: int,
                            rmax: int):
    """
    Sample |G| along ray x(r)=r*direction mod L, r=0..rmax.
    Returns vals[r] and step_norm=||direction||_2 (so you can convert "per-step" to "per-euclidean").
    """
    dirv = torch.tensor(direction, device=G.device, dtype=torch.int64)
    if dirv.numel() != d:
        raise ValueError(f"direction must have length d={d}, got {dirv.numel()}")
    step_norm = float(torch.sqrt(torch.sum(dirv.to(torch.float64)**2)).item())
    pos = torch.zeros((d,), device=G.device, dtype=torch.int64)
    vals = []
    for _ in range(rmax + 1):
        vals.append(torch.abs(G[tuple(pos.tolist())]))
        pos = (pos + dirv) % L
    return torch.stack(vals, dim=0), step_norm

class MassMeasurer:
    """
    Mass estimate from correlators by:
      - take |C|
      - prefactor-correct (multiply by (r+1)^{(d-1)/2} by default)
      - take monotone tail envelope
      - take log-slopes
      - pick a stable plateau window (min MAD)
    """

    @staticmethod
    def from_1d(vals,
                d: int,
                rmin: float,
                rmax: float,
                prefactor_power: float = None,
                win: int = 8):
        vals = _to_torch(vals)
        if prefactor_power is None:
            prefactor_power = 0.5 * (d - 1)

        corrected = prefactor_correct(torch.abs(vals), prefactor_power)
        env = tail_envelope_abs(corrected)
        mids, slopes = local_slopes_from_env(env)
        m_hat, mad, rng = robust_plateau(slopes, mids, rmin=rmin, rmax=rmax, win=win)
        return dict(m_hat=m_hat, mad=mad, plateau_mids=rng, env=env.detach().cpu(), mids=mids.detach().cpu(), slopes=slopes.detach().cpu())

    @staticmethod
    def from_torus_shell(Gx,
                         L: int,
                         d: int,
                         rmin: float,
                         rmax: float,
                         prefactor_power: float = None,
                         win: int = 8):
        """
        Shell version: env[r]=max_{L1(x)=r} |G(x)|.
        """
        Gx = _to_torch(Gx)
        if prefactor_power is None:
            prefactor_power = 0.5 * (d - 1)

        dist = torus_L1_dist_grid(L, d, device=Gx.device)
        r_max_total = int(d * (L // 2))
        env = env_by_dist_max(torch.abs(Gx), dist, r_max_total)

        corrected_env = prefactor_correct(env, prefactor_power)
        env2 = tail_envelope_abs(corrected_env)
        mids, slopes = local_slopes_from_env(env2)
        m_hat, mad, rng = robust_plateau(slopes, mids, rmin=rmin, rmax=rmax, win=win)
        return dict(m_hat=m_hat, mad=mad, plateau_mids=rng, env=env.detach().cpu(), mids=mids.detach().cpu(), slopes=slopes.detach().cpu())

    @staticmethod
    def from_torus_directions(Gx,
                              L: int,
                              d: int,
                              directions,
                              rmin: float,
                              rmax: float,
                              rmax_dir: int = None,
                              prefactor_power: float = None,
                              win: int = 8,
                              convert_to_euclidean: bool = True):
        """
        Directional version: sample along a few rays and combine.
        """
        Gx = _to_torch(Gx)
        if prefactor_power is None:
            prefactor_power = 0.5 * (d - 1)
        if rmax_dir is None:
            rmax_dir = max(1, L//2 - 2)

        per = []
        for direction in directions:
            vals, step_norm = directional_profile_abs(Gx, direction, L=L, d=d, rmax=rmax_dir)
            res = MassMeasurer.from_1d(vals, d=d, rmin=rmin, rmax=rmax, prefactor_power=prefactor_power, win=win)
            m_step = res["m_hat"]
            m_eucl = m_step / step_norm if convert_to_euclidean and step_norm > 0 else m_step
            per.append((tuple(direction), step_norm, m_eucl, m_step, res))

        masses = torch.tensor([x[2] for x in per], dtype=torch.float64)
        m_med = float(torch.median(masses).item())
        mad = float(torch.median(torch.abs(masses - torch.median(masses))).item())
        return dict(m_median=m_med, mad=mad, per_direction=per)

    @staticmethod
    def cosh_meff(Ct, tmin: int = 2, tmax: int = None):
        """
        Periodic-time effective mass:
          m_eff(t) = arcosh( (C(t-1)+C(t+1))/(2C(t)) )
        """
        Ct = _to_torch(Ct, device="cpu")
        T = Ct.numel()
        if tmax is None:
            tmax = T - 2
        ts, meff = [], []
        for t in range(tmin, tmax + 1):
            num = Ct[(t-1) % T] + Ct[(t+1) % T]
            den = 2.0 * Ct[t % T]
            x = float((num/den).clamp(min=1.0 + 1e-12).item())
            ts.append(t)
            meff.append(math.acosh(x))
        return np.array(ts), np.array(meff)

# ---- quick sanity demo (synthetic)
def _demo_mass():
    print("\n[DEMO] Mass module on synthetic 4D Yukawa tail")
    m_true, d = 0.55, 4
    rmax = 60
    r = torch.arange(rmax + 1, dtype=torch.float64)
    G = (r + 1.0) ** (-0.5*(d-1)) * torch.exp(-m_true * r)
    G = G * (1.0 + 0.02 * torch.randn_like(G))
    out = MassMeasurer.from_1d(G, d=d, rmin=10, rmax=40, win=10)
    print(f"  true m={m_true:.6f}   measured m={out['m_hat']:.6f}  (MAD~{out['mad']:.2e}, window {out['plateau_mids']})")

# ============================================================
# 2) SU(2) RG-INTERTWINING TEST (JAX)
# ============================================================

try:
    import jax
    import jax.numpy as jnp
    from functools import partial
except Exception as e:
    jax = None
    print("\n[WARN] JAX+JAXLIB not available => SU(2)/SU(3) parts will be skipped.\n", e)

if jax is not None:
    # ---- SU(2) quaternion ops
    def quat_mul(q1, q2):
        w1,x1,y1,z1 = jnp.split(q1, 4, axis=-1)
        w2,x2,y2,z2 = jnp.split(q2, 4, axis=-1)
        return jnp.concatenate([
            w1*w2 - x1*x2 - y1*y2 - z1*z2,
            w1*x2 + x1*w2 + y1*z2 - z1*y2,
            w1*y2 - x1*z2 + y1*w2 + z1*x2,
            w1*z2 + x1*y2 - y1*x2 + z1*w2
        ], axis=-1)

    def quat_inv(q):
        return jnp.concatenate([q[..., :1], -q[..., 1:]], axis=-1)

    def quat_unit(q):
        return q / (jnp.sqrt(jnp.sum(q*q, axis=-1, keepdims=True)) + 1e-12)

    def su2_exp(x):
        theta = jnp.linalg.norm(x, axis=-1, keepdims=True)
        half = 0.5 * theta
        w = jnp.cos(half)
        s = jnp.where(theta > 1e-12, jnp.sin(half)/theta, 0.5 - (theta*theta)/48.0)
        xyz = s * x
        return jnp.concatenate([w, xyz], axis=-1)

    def su2_log(q):
        q = quat_unit(q)
        w = q[..., :1]
        v = q[..., 1:]
        nv = jnp.linalg.norm(v, axis=-1, keepdims=True)
        ang = 2.0 * jnp.arctan2(nv, w)
        coef = jnp.where(nv > 1e-12, ang/nv, 2.0/(w + 1e-12))
        return coef * v

    def karcher_mean(quats, iters=5):
        q0 = quat_unit(quats[0])
        def body(_, qcur):
            logs = su2_log(quat_mul(quat_inv(qcur), quats))
            delta = jnp.mean(logs, axis=0)
            return quat_unit(quat_mul(qcur, su2_exp(delta)))
        return jax.lax.fori_loop(0, iters, body, q0)

    def pi_geodesic(U_block, iters=5):
        return su2_log(karcher_mean(U_block, iters=iters))  # R^3

    def compute_R_coord(U_block, v, iters=5):
        # coordinate gradient in raw quaternion coords (factor-of-4 trap)
        def F_pullback(flatU):
            U = flatU.reshape((-1,4))
            Y = pi_geodesic(U, iters=iters)
            return jnp.dot(v, Y)
        g = jax.grad(F_pullback)(U_block.reshape(-1)).reshape((-1,4))[:,1:]
        return jnp.sum(g*g) / (jnp.sum(v*v) + 1e-12)

    def compute_R_intrinsic(U_block, v, iters=5):
        # intrinsic gradient in Lie algebra coords (no factor-of-4 trap)
        n_links = U_block.shape[0]
        X0 = jnp.zeros((n_links,3), dtype=U_block.dtype)
        def F_pullback(X):
            U_pert = quat_mul(U_block, su2_exp(X))  # right-multiply by exp(X)
            Y = pi_geodesic(U_pert, iters=iters)
            return jnp.dot(v, Y)
        gX = jax.grad(F_pullback)(X0)
        return jnp.sum(gX*gX) / (jnp.sum(v*v) + 1e-12)

    @partial(jax.jit, static_argnames=("n_samples","n_links","iters"))
    def batch_R(key, n_samples: int = 512, n_links: int = 16, eps: float = 0.15, iters: int = 5):
        """
        FIXED: static_argnames (keyword-safe) so n_samples can safely control shapes.
        """
        keys = jax.random.split(key, n_samples)
        def one(k):
            k1, k2 = jax.random.split(k)
            X = eps * jax.random.normal(k1, (n_links, 3))
            U = su2_exp(X)
            v = jax.random.normal(k2, (3,))
            v = v / (jnp.linalg.norm(v) + 1e-12)
            return compute_R_intrinsic(U, v, iters=iters), compute_R_coord(U, v, iters=iters)
        Rin, Rco = jax.vmap(one)(keys)
        return Rin, Rco

    def run_rg_test():
        print("\n[DEMO] SU(2) RG intertwining: intrinsic vs coordinate gradients")
        Rin, Rco = batch_R(jax.random.PRNGKey(0), n_samples=256, n_links=16, eps=0.15, iters=5)
        Rin = np.array(Rin); Rco = np.array(Rco)
        print(f"  intrinsic R: mean={Rin.mean():.6g} median={np.median(Rin):.6g} max={Rin.max():.6g}")
        print(f"  coord     R: mean={Rco.mean():.6g} median={np.median(Rco):.6g} max={Rco.max():.6g}")
        print(f"  coord/intrinsic median ratio ≈ {np.median(Rco)/np.median(Rin):.3g} (≈4 means you hit the quaternion-coordinate trap)")

# ============================================================
# 3) CASRG MODEL2: cheaper eigensolvers (LOBPCG / IRL / shift-invert)
# ============================================================

def symmetrize_sparse(H: torch.Tensor) -> torch.Tensor:
    """(H + H^T)/2 for sparse COO."""
    Hs = (H + H.transpose(0,1)) * 0.5
    return Hs.coalesce()

def lowest_eigs_lobpcg(H: torch.Tensor, k: int = 4, tol: float = 1e-8, niter: int = 250, seed: int = 0):
    """
    Compute k smallest eigenvalues with torch.lobpcg (GPU-friendly).
    H must be symmetric and sparse.
    """
    H = symmetrize_sparse(H)
    n = H.shape[0]
    gen = torch.Generator(device=H.device)
    gen.manual_seed(seed)
    X = torch.randn((n,k), device=H.device, dtype=H.dtype, generator=gen)
    X, _ = torch.linalg.qr(X, mode="reduced")
    evals, _ = torch.lobpcg(H, k=k, largest=False, tol=tol, niter=niter, init=X)
    evals = evals.detach().cpu().numpy()
    evals.sort()
    return evals

def torch_sparse_to_scipy_csr(H: torch.Tensor):
    import scipy.sparse as sp
    H = H.coalesce().cpu()
    idx = H.indices().numpy()
    dat = H.values().numpy()
    return sp.coo_matrix((dat, (idx[0], idx[1])), shape=H.shape).tocsr()

def lowest_eigs_eigsh(H: torch.Tensor, k: int = 4, which: str = "SA", tol: float = 1e-10, maxiter: int = 5000):
    """
    CPU IRL (scipy.sparse.linalg.eigsh). Reliable, but moves H to CPU.
    """
    import scipy.sparse.linalg as spla
    Hcsr = torch_sparse_to_scipy_csr(H)
    evals, _ = spla.eigsh(Hcsr, k=k, which=which, tol=tol, maxiter=maxiter)
    evals = np.array(evals, dtype=float)
    evals.sort()
    return evals

def lowest_eigs_shift_invert(H: torch.Tensor, k: int = 4, sigma: float = 0.0, tol: float = 1e-10, maxiter: int = 5000):
    """
    CPU shift-invert around sigma using scipy.eigsh(..., sigma=sigma).
    Good if you need very accurate low-lying spectrum and H is ill-conditioned.
    """
    import scipy.sparse.linalg as spla
    Hcsr = torch_sparse_to_scipy_csr(H)
    evals, _ = spla.eigsh(Hcsr, k=k, sigma=sigma, which="LM", tol=tol, maxiter=maxiter)
    evals = np.array(evals, dtype=float)
    evals.sort()
    return evals

def gap_from_eigs(evals):
    evals = np.array(evals, dtype=float)
    evals.sort()
    return float(evals[0]), float(evals[1] - evals[0])

def casrg_gap_scan(build_H_fn,
                   eps_bits,
                   truncations,
                   Ls,
                   solver: str = "lobpcg",
                   eig_k: int = 4):
    """
    build_H_fn(L, trunc, eps_bit) -> torch sparse COO (symmetric)
    Returns list of dicts with (L,trunc,eps_bit,e0,gap,evals).
    """
    out = []
    for L in Ls:
        for trunc in truncations:
            for eps in eps_bits:
                H = build_H_fn(L=L, trunc=trunc, eps_bit=eps).to(device=TORCH_DEVICE, dtype=TORCH_DTYPE)
                if solver == "lobpcg":
                    evals = lowest_eigs_lobpcg(H, k=eig_k)
                elif solver == "eigsh":
                    evals = lowest_eigs_eigsh(H, k=eig_k)
                elif solver == "shift-invert":
                    evals = lowest_eigs_shift_invert(H, k=eig_k, sigma=0.0)
                else:
                    raise ValueError("solver must be 'lobpcg', 'eigsh', or 'shift-invert'")
                e0, gap = gap_from_eigs(evals)
                out.append(dict(L=L, trunc=trunc, eps_bit=eps, e0=e0, gap=gap, evals=evals))
                print(f"[CasRG] L={L} trunc={trunc} eps_bit={eps:.3g}  e0={e0:.6g} gap={gap:.6g}")
    return out

def print_casrg_patch_notes():
    print(r"""
[PATCH NOTES] CasRG model2 eigen solver swap

Where you currently run full-reorth Lanczos (slow):
    evals = lanczos_sparse_full_reorth(Hs)

Swap to:
    evals = lowest_eigs_lobpcg(Hs, k=4, tol=1e-8, niter=250)

Then do a modest scan like:
    results = casrg_gap_scan(build_H_model2, eps_bits=[0,1e-4,1e-3,1e-2],
                             truncations=[J1,J2], Ls=[2,3], solver="lobpcg")

If the lowest spectrum is *very* clustered / ill-conditioned, use IRL or shift-invert:
    solver="eigsh"     (robust)
    solver="shift-invert"  (most accurate, CPU + factorization)
""")

# ============================================================
# 4) SU(3) COMMUTATOR SEED UPGRADE (JAX)
# ============================================================

if jax is not None:
    def build_su3_f_tensor():
        f = np.zeros((8,8,8), dtype=float)
        entries = [
            (1,2,3, 1.0),
            (1,4,7, 0.5),
            (1,5,6, 0.5),
            (2,4,6, 0.5),
            (2,5,7,-0.5),
            (3,4,5, 0.5),
            (3,6,7, 0.5),
            (4,5,8, math.sqrt(3)/2),
            (6,7,8, math.sqrt(3)/2),
        ]
        for (a,b,c,val) in entries:
            a-=1; b-=1; c-=1
            f[a,b,c] =  val
            f[b,c,a] =  val
            f[c,a,b] =  val
            f[a,c,b] = -val
            f[c,b,a] = -val
            f[b,a,c] = -val
        return jnp.array(f)

    F_SU3 = build_su3_f_tensor()
    M_SU3 = jnp.sum(F_SU3 * F_SU3, axis=0)  # M_bc = Σ_a f[a,b,c]^2

    def su3_comm_true_pair(A, B):
        comm = jnp.einsum("abc,...b,...c->...a", F_SU3, A, B)
        return jnp.sum(comm*comm, axis=-1)

    def su3_comm_surrogate_pair(A, B):
        A2 = A*A
        B2 = B*B
        return jnp.einsum("...b,...c,bc->...", A2, B2, M_SU3)

    def su3_comm_seed_true(theta_mu, kappa: float):
        """
        TRUE: kappa * Σ_{mu<nu} ||[A_mu, A_nu]||^2
        theta_mu: (...,4,8)
        """
        total = 0.0
        for mu in range(4):
            for nu in range(mu+1, 4):
                total = total + su3_comm_true_pair(theta_mu[...,mu,:], theta_mu[...,nu,:])
        return kappa * total

    def su3_comm_seed_surrogate(theta_mu, kappa: float):
        """
        SURROGATE: kappa * Σ_{mu<nu} Σ_{b,c} M_bc (A_mu_b^2)(A_nu_c^2)
        theta_mu: (...,4,8)
        """
        total = 0.0
        for mu in range(4):
            for nu in range(mu+1, 4):
                total = total + su3_comm_surrogate_pair(theta_mu[...,mu,:], theta_mu[...,nu,:])
        return kappa * total

    # ---- tiny Wilson single-site + Hessian compare (proxy for your full convexity engine)
    def gell_mann():
        i    = 0.0 + 1.0j
        lam = []
        lam.append(jnp.array([[0,1,0],[1,0,0],[0,0,0]], dtype=jnp.complex64))
        lam.append(jnp.array([[0,-i,0],[i,0,0],[0,0,0]], dtype=jnp.complex64))
        lam.append(jnp.array([[1,0,0],[0,-1,0],[0,0,0]], dtype=jnp.complex64))
        lam.append(jnp.array([[0,0,1],[0,0,0],[1,0,0]], dtype=jnp.complex64))
        lam.append(jnp.array([[0,0,-i],[0,0,0],[i,0,0]], dtype=jnp.complex64))
        lam.append(jnp.array([[0,0,0],[0,0,1],[0,1,0]], dtype=jnp.complex64))
        lam.append(jnp.array([[0,0,0],[0,0,-i],[0,i,0]], dtype=jnp.complex64))
        lam.append((1.0/math.sqrt(3.0))*jnp.array([[1,0,0],[0,1,0],[0,0,-2]], dtype=jnp.complex64))
        return jnp.stack(lam, axis=0)

    LAM = gell_mann()
    T_BASIS = (0.5j) * LAM  # anti-Hermitian basis

    def su3_alg_from_vec(theta):
        return jnp.einsum("...a,aij->...ij", theta, T_BASIS)

    def su3_exp(theta):
        import jax.scipy.linalg as jsp
        return jsp.expm(su3_alg_from_vec(theta))

    def su3_dag(U):
        return jnp.swapaxes(jnp.conjugate(U), -1, -2)

    def wilson_single_site(theta_mu, beta: float):
        U = jax.vmap(su3_exp)(theta_mu)  # (4,3,3)
        S = 0.0
        for mu in range(4):
            for nu in range(mu+1, 4):
                P = U[mu] @ U[nu] @ su3_dag(U[mu]) @ su3_dag(U[nu])
                S = S + (1.0 - jnp.real(jnp.trace(P))/3.0)
        return beta * S

    def action_seeded(theta_flat, beta: float, kappa: float, which: str):
        theta_mu = theta_flat.reshape((4,8))
        m2 = 0.2
        base = wilson_single_site(theta_mu, beta=beta) + 0.5*m2*jnp.sum(theta_mu*theta_mu)
        if which == "true":
            return base + su3_comm_seed_true(theta_mu, kappa=kappa)
        if which == "surrogate":
            return base + su3_comm_seed_surrogate(theta_mu, kappa=kappa)
        raise ValueError

    def hessian_spectrum(action_fn, x):
        H = jax.hessian(action_fn)(x)
        evals = jnp.linalg.eigvalsh(H)
        return evals

    def safe_radius_grid(action_fn, dim: int, key, eps_grid, n_dirs: int = 12):
        keys = jax.random.split(key, n_dirs)
        def one(k):
            u = jax.random.normal(k, (dim,))
            u = u / (jnp.linalg.norm(u) + 1e-12)
            safe = 0.0
            for eps in eps_grid:
                evals = hessian_spectrum(action_fn, eps*u)
                safe = jnp.where(evals[0] > 0.0, eps, safe)
            return safe
        return np.array(jax.vmap(one)(keys))

    def demo_su3_seed():
        print("\n[DEMO] SU(3) commutator seed: surrogate vs true (Hessian + safe-radius proxy)")
        beta = 0.20
        kappa = 0.50
        dim = 32

        a_true = lambda z: action_seeded(z, beta=beta, kappa=kappa, which="true")
        a_sur  = lambda z: action_seeded(z, beta=beta, kappa=kappa, which="surrogate")

        key = jax.random.PRNGKey(123)
        u = jax.random.normal(key, (dim,))
        u = u / (jnp.linalg.norm(u) + 1e-12)
        eps = 0.35

        evals_true = np.array(hessian_spectrum(a_true, eps*u))
        evals_sur  = np.array(hessian_spectrum(a_sur,  eps*u))
        print(f"  At eps={eps:.3f}: min eig true={evals_true[0]:.6g}, surrogate={evals_sur[0]:.6g}")
        print(f"    true eig[0:6]={evals_true[:6]}")
        print(f"    sur  eig[0:6]={evals_sur[:6]}")

        eps_grid = np.linspace(0.0, 0.8, 21)
        safe_true = safe_radius_grid(a_true, dim, jax.random.PRNGKey(1), eps_grid, n_dirs=10)
        safe_sur  = safe_radius_grid(a_sur,  dim, jax.random.PRNGKey(2), eps_grid, n_dirs=10)
        print(f"  grid-safe eps (10 dirs): median true={np.median(safe_true):.3f}, surrogate={np.median(safe_sur):.3f}")
        print("  (This is a single-site proxy; your full lattice safe region will be smaller.)")

    def print_su3_swap_notes():
        print(r"""
[PATCH NOTES] SU(3) commutator seed swap

Where you currently use a 'square-of-squares' surrogate, replace it with:

    seed = su3_comm_seed_true(theta_mu, kappa)

with theta_mu shaped (...,4,8) (four directions, adjoint coords).

That computes:
    kappa * Σ_{mu<nu} ||[A_mu, A_nu]||^2
with commutator components:
    [A_mu, A_nu]_a = Σ_{b,c} f_{a b c} A_mu^b A_nu^c

Then re-run your Hessian-spectrum + safe-region scan to compare against:
    su3_comm_seed_surrogate(theta_mu, kappa)
""")

# ============================================================
# RUN (set these flags)
# ============================================================

RUN_MASS_DEMO = True
RUN_RG_DEMO   = True
RUN_SU3_DEMO  = True

if RUN_MASS_DEMO:
    _demo_mass()

print_casrg_patch_notes()

if jax is not None and RUN_RG_DEMO:
    run_rg_test()

if jax is not None:
    print_su3_swap_notes()
    if RUN_SU3_DEMO:
        demo_su3_seed()

# ============================================================
# HOW TO USE ON YOUR DATA (quick recipes)
# ============================================================
#
# (A) Interacting correlator on T^4 (gauge-projected kernel, etc):
#     Gx = ...  # numpy/torch array shape [L,L,L,L], translation invariant
#     res = MassMeasurer.from_torus_shell(Gx, L=L, d=4, rmin=10, rmax=30)
#     print(res["m_hat"])
#
# (B) Directional masses:
#     dirs = [(1,0,0,0),(1,1,0,0),(1,1,1,0),(1,1,1,1)]
#     res = MassMeasurer.from_torus_directions(Gx, L=L, d=4, directions=dirs,
#                                              rmin=6, rmax=20, rmax_dir=L//2-2)
#     print(res["m_median"], res["mad"])
#
# (C) Wilson-loop / temporal correlator C(t) (periodic):
#     ts, meff = MassMeasurer.cosh_meff(Ct)
#     # then plateau-pick on meff if you like, or do tail-envelope slopes with prefactor_power=0:
#     out = MassMeasurer.from_1d(Ct, d=1, rmin=4, rmax=12, prefactor_power=0.0)
#     print("V(R)~", out["m_hat"])
#
# (D) CasRG model2 gap scan:
#     def build_H_model2(L, trunc, eps_bit): ...
#     results = casrg_gap_scan(build_H_model2,
#                              eps_bits=[0.0,1e-4,1e-3,1e-2,1e-1],
#                              truncations=[...],
#                              Ls=[2,3],
#                              solver="lobpcg")
# ============================================================



[DEMO] Mass module on synthetic 4D Yukawa tail
  true m=0.550000   measured m=0.556540  (MAD~9.21e-03, window (21.5, 30.5))

[PATCH NOTES] CasRG model2 eigen solver swap

Where you currently run full-reorth Lanczos (slow):
    evals = lanczos_sparse_full_reorth(Hs)

Swap to:
    evals = lowest_eigs_lobpcg(Hs, k=4, tol=1e-8, niter=250)

Then do a modest scan like:
    results = casrg_gap_scan(build_H_model2, eps_bits=[0,1e-4,1e-3,1e-2],
                             truncations=[J1,J2], Ls=[2,3], solver="lobpcg")

If the lowest spectrum is *very* clustered / ill-conditioned, use IRL or shift-invert:
    solver="eigsh"     (robust)
    solver="shift-invert"  (most accurate, CPU + factorization)


[DEMO] SU(2) RG intertwining: intrinsic vs coordinate gradients
  intrinsic R: mean=nan median=nan max=nan
  coord     R: mean=nan median=nan max=nan
  coord/intrinsic median ratio ≈ nan (≈4 means you hit the quaternion-coordinate trap)

[PATCH NOTES] SU(3) commutator seed swap

Where you curren

/usr/local/lib/python3.12/dist-packages/jax/_src/lax/lax.py:5473: ComplexWarning: Casting complex values to real discards the imaginary part
  x_bar = _convert_element_type(x_bar, x.aval.dtype, x.aval.weak_type)


  At eps=0.350: min eig true=0.17211, surrogate=0.19841
    true eig[0:6]=[0.17210972 0.17565002 0.18067285 0.18342575 0.19575062 0.20200203]
    sur  eig[0:6]=[0.19841032 0.20302065 0.20454891 0.20588928 0.21531667 0.21687661]
  grid-safe eps (10 dirs): median true=0.800, surrogate=0.800
  (This is a single-site proxy; your full lattice safe region will be smaller.)


In [2]:
import math
import numpy as np
import warnings
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, Any, Callable

# ============================================================
# (1) MASS MODULE: prefactor-corrected plateau extractor
# ============================================================

def mad(x: np.ndarray) -> float:
    """Median absolute deviation (MAD), ignoring NaNs."""
    x = np.asarray(x, dtype=float)
    med = np.nanmedian(x)
    return float(np.nanmedian(np.abs(x - med)))

@dataclass
class MassEstimate:
    m: float                     # mass in physical-distance units
    mad: float                   # MAD within chosen plateau window
    window: Tuple[float, float]  # plateau window in r_mid units
    r_mid: np.ndarray            # midpoints where m_eff is defined
    m_eff: np.ndarray            # local effective masses
    details: Dict[str, Any]      # diagnostics

def prefactor_corrected_y(C: np.ndarray, r_phys: np.ndarray, d: int, eps: float = 1e-300) -> np.ndarray:
    """
    y(r) = log |C(r)| + ((d-1)/2) log r.

    If C(r) ~ A * exp(-m r) / r^{(d-1)/2} in d dims,
    then y(r) ~ const - m r, so slopes of y vs r give m.
    """
    C = np.asarray(C)
    r = np.asarray(r_phys, dtype=float)
    power = 0.5 * (d - 1)
    rr = np.maximum(r, 1e-12)  # avoid log(0)
    return np.log(np.maximum(np.abs(C), eps)) + power * np.log(rr)

def local_linear_slopes(y: np.ndarray, r: np.ndarray, W: int) -> Tuple[np.ndarray, np.ndarray]:
    """
    Slide a window of length (W+1) and fit y ≈ a + b r in each window.
    Returns:
      r_mid[i] = 0.5*(r[i] + r[i+W])
      m_eff[i] = -b_i
    """
    y = np.asarray(y, dtype=float)
    r = np.asarray(r, dtype=float)
    n = len(r)
    if n < W + 2:
        raise ValueError(f"Need at least {W+2} points, got {n}")

    r_mid = np.empty(n - W, dtype=float)
    m_eff = np.empty(n - W, dtype=float)

    for i in range(0, n - W):
        rr = r[i:i+W+1]
        yy = y[i:i+W+1]
        if not (np.all(np.isfinite(rr)) and np.all(np.isfinite(yy))):
            r_mid[i] = np.nan
            m_eff[i] = np.nan
            continue
        A = np.vstack([rr, np.ones_like(rr)]).T
        b, a = np.linalg.lstsq(A, yy, rcond=None)[0]
        r_mid[i] = 0.5 * (rr[0] + rr[-1])
        m_eff[i] = -b

    return r_mid, m_eff

def pick_plateau_window(r_mid: np.ndarray,
                        m_eff: np.ndarray,
                        win: int = 10,
                        r_min: Optional[float] = None,
                        r_max: Optional[float] = None,
                        require_positive: bool = True) -> Tuple[slice, float, float]:
    """
    Pick a contiguous window (length=win) of m_eff with minimal MAD.
    """
    r_mid = np.asarray(r_mid, dtype=float)
    m_eff = np.asarray(m_eff, dtype=float)

    mask = np.isfinite(r_mid) & np.isfinite(m_eff)
    if r_min is not None:
        mask &= (r_mid >= r_min)
    if r_max is not None:
        mask &= (r_mid <= r_max)

    if np.count_nonzero(mask) < win:
        raise ValueError(f"Not enough valid points for plateau win={win} (have {np.count_nonzero(mask)})")

    best = None
    for start in range(0, len(r_mid) - win + 1):
        sl = slice(start, start + win)
        if not np.all(mask[sl]):
            continue
        seg = m_eff[sl]
        med = float(np.nanmedian(seg))
        if require_positive and not (med > 0.0):
            continue
        this_mad = mad(seg)
        score = this_mad  # simplest flatness criterion
        if best is None or score < best[0]:
            best = (score, sl, med, float(this_mad))

    if best is None:
        raise ValueError("Could not find a plateau window satisfying constraints.")
    _, sl, med, this_mad = best
    return sl, med, this_mad

def measure_mass_yukawa_prefactor(C: np.ndarray,
                                 r_steps: np.ndarray,
                                 d: int = 4,
                                 step_norm: float = 1.0,
                                 W: int = 9,
                                 plateau_win: int = 10,
                                 r_min: Optional[float] = None,
                                 r_max: Optional[float] = None) -> MassEstimate:
    """
    Mass measurement for Yukawa-like tails in d dimensions.

    Inputs:
      C        : correlator samples vs step index (or distance)
      r_steps  : step index 0,1,2,... OR your bin centers
      step_norm: physical distance per step (e.g. ||step||_2 for a lattice ray)

    Output mass 'm' is per *physical distance* (i.e. in units of 1/(step_norm)).
    """
    r_steps = np.asarray(r_steps, dtype=float)
    r_phys = step_norm * r_steps

    y = prefactor_corrected_y(C, r_phys, d=d)
    r_mid, m_eff = local_linear_slopes(y, r_phys, W=W)

    # Default range: avoid the very-short-distance transient and the wrap/finite-size region.
    if r_min is None:
        r_min = float(np.nanmin(r_mid) + 0.2 * (np.nanmax(r_mid) - np.nanmin(r_mid)))
    if r_max is None:
        r_max = float(np.nanmin(r_mid) + 0.85 * (np.nanmax(r_mid) - np.nanmin(r_mid)))

    sl, med, this_mad = pick_plateau_window(r_mid, m_eff, win=plateau_win, r_min=r_min, r_max=r_max)
    window = (float(r_mid[sl.start]), float(r_mid[sl.stop - 1]))

    return MassEstimate(
        m=float(med),
        mad=float(this_mad),
        window=window,
        r_mid=r_mid,
        m_eff=m_eff,
        details=dict(W=W, plateau_win=plateau_win, r_min=r_min, r_max=r_max, step_norm=step_norm),
    )

def ray_extract_torus(G: np.ndarray, step: Tuple[int, ...], n_max: Optional[int] = None) -> Tuple[np.ndarray, np.ndarray]:
    """
    Extract values along the torus ray x = n*step mod L from a d-dim array G shaped (L,)*d.
    Returns (n_steps, vals).
    """
    G = np.asarray(G)
    d = G.ndim
    step = np.asarray(step, dtype=int)
    if len(step) != d:
        raise ValueError("step must have length equal to G.ndim")

    L = G.shape[0]
    if not all(s == L for s in G.shape):
        raise ValueError("G must be on a cubic torus: shape (L,)*d")

    if n_max is None:
        n_max = L // 2

    ns = np.arange(n_max + 1, dtype=int)
    coords = (ns[:, None] * step[None, :]) % L
    vals = np.array([G[tuple(c)] for c in coords], dtype=G.dtype)
    return ns.astype(float), vals

def demo_mass_module_synth_4d(seed: int = 0):
    rng = np.random.default_rng(seed)
    true_m = 0.55
    d = 4
    r = np.arange(1, 51, dtype=float)
    A = 1.0
    C_clean = A * np.exp(-true_m * r) / (r ** ((d - 1) / 2))
    noise = 0.03
    C_noisy = C_clean * (1.0 + noise * rng.normal(size=r.shape))

    est = measure_mass_yukawa_prefactor(C_noisy, r_steps=r, d=d, step_norm=1.0, W=9, plateau_win=10)

    print("[DEMO] Mass module on synthetic 4D Yukawa tail")
    print(f"  true m={true_m:.6f}   measured m={est.m:.6f}  (MAD~{est.mad:.2e}, window {est.window})")
    return est

# -----------------------------
# YOUR DATA HOOKS (mass module)
# -----------------------------
# Case A: you already have a correlator vs distance bins:
#   est = measure_mass_yukawa_prefactor(C=Corr, r_steps=r_bins, d=4, step_norm=1.0)
#
# Case B: you have a full torus kernel K[x] and want a directional mass:
#   step = (1,1,0,0)              # for example
#   n, vals = ray_extract_torus(K, step=step, n_max=K.shape[0]//2)
#   est = measure_mass_yukawa_prefactor(vals[1:], r_steps=n[1:], d=4, step_norm=np.linalg.norm(step))
# (drop n=0 if your correlator has a contact term)


# ============================================================
# (2) SU(2) RG INTERTWINING: intrinsic tangent norm + static-arg fix
# ============================================================

def _try_import_jax():
    try:
        import jax
        import jax.numpy as jnp
        from jax import random
        return jax, jnp, random
    except Exception:
        return None, None, None

jax, jnp, random = _try_import_jax()

if jax is not None:
    from functools import partial

    # ---- quaternion ops: q = (w,x,y,z) ----
    def qnormalize(q):
        q = jnp.asarray(q)
        return q / jnp.linalg.norm(q, axis=-1, keepdims=True)

    def qconj(q):
        q = jnp.asarray(q)
        return jnp.concatenate([q[..., :1], -q[..., 1:]], axis=-1)

    def qmul(a, b):
        aw, av = a[..., :1], a[..., 1:]
        bw, bv = b[..., :1], b[..., 1:]
        w = aw * bw - jnp.sum(av * bv, axis=-1, keepdims=True)
        v = aw * bv + bw * av + jnp.cross(av, bv)
        return jnp.concatenate([w, v], axis=-1)

    def qinv(q):
        return qconj(q) / jnp.sum(q * q, axis=-1, keepdims=True)

    def canonicalize(q):
        # choose SU(2) representative with w>=0 to avoid antipodal sign flips
        q = qnormalize(q)
        s = jnp.where(q[..., 0:1] < 0, -1.0, 1.0)
        return q * s

    # exp/log with the standard SU(2) half-angle convention
    def su2_exp(x):
        x = jnp.asarray(x)
        theta = jnp.linalg.norm(x, axis=-1, keepdims=True)
        half = 0.5 * theta
        w = jnp.cos(half)
        s_over_theta = jnp.where(theta > 1e-10, jnp.sin(half) / theta, 0.5 - (theta ** 2) / 48.0)
        v = s_over_theta * x
        return canonicalize(jnp.concatenate([w, v], axis=-1))

    def su2_log(q):
        q = canonicalize(q)
        w = jnp.clip(q[..., 0:1], -1.0, 1.0)
        v = q[..., 1:]
        s = jnp.linalg.norm(v, axis=-1, keepdims=True)
        theta = 2.0 * jnp.arctan2(s, w)
        theta_over_s = jnp.where(s > 1e-10, theta / s, 2.0 + 0.0 * s)
        return theta_over_s * v

    def karcher_mean(U, iters: int = 3):
        U = canonicalize(U)
        q = canonicalize(jnp.mean(U, axis=0))
        for _ in range(iters):
            dots = jnp.sum(U * q[None, :], axis=-1, keepdims=True)
            U_al = jnp.where(dots < 0.0, -U, U)
            deltas = su2_log(qmul(qinv(q), U_al))     # (n,3)
            delta = jnp.mean(deltas, axis=0)          # (3,)
            q = canonicalize(qmul(q, su2_exp(delta)))
        return q

    def pi_geodesic(U_block, iters: int = 3):
        qbar = karcher_mean(U_block, iters=iters)
        return su2_log(qbar)  # coarse algebra coord (3,)

    def intrinsic_grad_norm_sq(U_block, gradU):
        """
        Intrinsic tangent norm (fixes the quaternion coordinate factor-of-4 trap):
          g_tan = g - (g·q) q
          xi    = 2 * Im(q^{-1} g_tan)
          ||grad||^2 = Σ_sites ||xi||^2
        """
        dot = jnp.sum(gradU * U_block, axis=-1, keepdims=True)
        g_tan = gradU - dot * U_block
        xi_half = qmul(qinv(U_block), g_tan)[..., 1:]
        xi = 2.0 * xi_half
        return jnp.sum(xi * xi)

    def compute_R_intrinsic(U_block, v, iters: int = 3):
        v = jnp.asarray(v)
        def F(flatU):
            U = flatU.reshape((16, 4))
            y = pi_geodesic(U, iters=iters)
            return jnp.dot(v, y)
        flat0 = U_block.reshape((-1,))
        grad_flat = jax.grad(F)(flat0).reshape((16, 4))
        num = intrinsic_grad_norm_sq(U_block, grad_flat)
        denom = jnp.sum(v * v)
        return num / denom

    def compute_R_coord(U_block, v, iters: int = 3):
        # coordinate version (intentionally "wrong" metric): drop scalar, no tangent projection, no factor-2
        v = jnp.asarray(v)
        def F(flatU):
            U = flatU.reshape((16, 4))
            y = pi_geodesic(U, iters=iters)
            return jnp.dot(v, y)
        flat0 = U_block.reshape((-1,))
        grad_flat = jax.grad(F)(flat0).reshape((16, 4))
        g_coord = grad_flat[:, 1:]
        num = jnp.sum(g_coord * g_coord)
        denom = jnp.sum(v * v)
        return num / denom

    # ---- The explicit static-arg fix (keyword-safe) ----
    # This is the version that prevents JAX from treating n_samples as traced when passed by keyword.
    @partial(jax.jit, static_argnames=("n_samples", "iters"))
    def batch_R_both(key, n_samples: int = 4096, eps: float = 0.1, iters: int = 3):
        """
        Returns arrays (R_intrinsic, R_coord) of length n_samples.
        static_argnames fixes the keyword/static mismatch that breaks random.split.
        """
        keys = random.split(key, n_samples)

        def one(k):
            k1, k2 = random.split(k)
            X = eps * random.normal(k1, (16, 3))
            U = su2_exp(X)
            v = random.normal(k2, (3,))
            return compute_R_intrinsic(U, v, iters=iters), compute_R_coord(U, v, iters=iters)

        out = jax.vmap(one)(keys)  # (n_samples,2)
        return out[0], out[1]

    def demo_su2_rg_intertwining(n_samples: int = 8192, eps: float = 0.1, iters: int = 3, seed: int = 0):
        key = random.PRNGKey(seed)
        R_intr, R_coord = batch_R_both(key, n_samples=n_samples, eps=eps, iters=iters)

        R_intr = np.array(R_intr)
        R_coord = np.array(R_coord)

        def stats(x):
            x = x[np.isfinite(x)]
            return dict(mean=float(np.mean(x)), median=float(np.median(x)), max=float(np.max(x)))

        si = stats(R_intr)
        sc = stats(R_coord)
        ratio_med = sc["median"] / si["median"] if (si["median"] != 0 and np.isfinite(si["median"])) else np.nan

        print("\n[DEMO] SU(2) RG intertwining: intrinsic vs coordinate gradients")
        print(f"  intrinsic R: mean={si['mean']:.6g} median={si['median']:.6g} max={si['max']:.6g}")
        print(f"  coord     R: mean={sc['mean']:.6g} median={sc['median']:.6g} max={sc['max']:.6g}")
        print(f"  coord/intrinsic median ratio ≈ {ratio_med:.6g} (≈4 means quaternion-coordinate trap)")
        return R_intr, R_coord

else:
    def demo_su2_rg_intertwining(*args, **kwargs):
        print("JAX not available; skipping SU(2) RG intertwining demo.")
        return None, None


# ============================================================
# (3) CasRG model2 eigen solver swap: LOBPCG / eigsh / shift-invert
# ============================================================

def _try_import_scipy():
    try:
        import scipy.sparse as sp
        import scipy.sparse.linalg as spla
        return sp, spla
    except Exception:
        return None, None

sp, spla = _try_import_scipy()

def lowest_eigs_lobpcg(H, k: int = 4, tol: float = 1e-8, niter: int = 250, seed: int = 0):
    """
    Lowest k eigenvalues using SciPy LOBPCG.
    Falls back to eigsh(which='SA') if LOBPCG chokes.
    """
    if spla is None:
        raise ImportError("SciPy sparse.linalg not available.")

    n = H.shape[0]
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, k))
    X, _ = np.linalg.qr(X)  # orthonormalize

    try:
        evals, _ = spla.lobpcg(H, X, largest=False, tol=tol, maxiter=niter)
        evals = np.asarray(evals, dtype=float)
        return np.sort(evals)
    except Exception as e:
        warnings.warn(f"lobpcg failed ({e}); falling back to eigsh(which='SA').")
        evals = spla.eigsh(H, k=k, which="SA", tol=tol, return_eigenvectors=False)
        return np.sort(np.asarray(evals, dtype=float))

def lowest_eigs_eigsh(H, k: int = 4, tol: float = 1e-10):
    if spla is None:
        raise ImportError("SciPy sparse.linalg not available.")
    evals = spla.eigsh(H, k=k, which="SA", tol=tol, return_eigenvectors=False)
    return np.sort(np.asarray(evals, dtype=float))

def lowest_eigs_shift_invert(H, k: int = 4, sigma: float = 0.0, tol: float = 1e-12):
    """
    Shift-invert: finds eigenvalues closest to sigma (default 0).
    More accurate but can be expensive due to factorization.
    """
    if spla is None:
        raise ImportError("SciPy sparse.linalg not available.")
    evals = spla.eigsh(H, k=k, sigma=sigma, which="LM", tol=tol, return_eigenvectors=False)
    return np.sort(np.asarray(evals, dtype=float))

def lowest_spectrum(H, solver: str = "lobpcg", k: int = 4, **solver_kwargs):
    solver = solver.lower()
    if solver == "lobpcg":
        return lowest_eigs_lobpcg(H, k=k, **solver_kwargs)
    if solver == "eigsh":
        return lowest_eigs_eigsh(H, k=k, **solver_kwargs)
    if solver in ("shift-invert", "shift_invert", "shiftinvert"):
        return lowest_eigs_shift_invert(H, k=k, **solver_kwargs)
    raise ValueError(f"Unknown solver='{solver}'")

def casrg_gap_scan(build_H_fn: Callable[..., Any],
                   eps_bits,
                   truncations,
                   Ls,
                   solver: str = "lobpcg",
                   k: int = 4,
                   solver_kwargs: Optional[dict] = None,
                   build_kwargs: Optional[dict] = None):
    """
    Generic scan wrapper.
    build_H_fn must return a symmetric SciPy sparse matrix (csr/csc) or LinearOperator.

    Expected usage:
        results = casrg_gap_scan(build_H_model2,
                                 eps_bits=[0,1e-4,1e-3,1e-2],
                                 truncations=[J1,J2],
                                 Ls=[2,3],
                                 solver="lobpcg")
    """
    if solver_kwargs is None: solver_kwargs = {}
    if build_kwargs is None: build_kwargs = {}

    out = {}
    for L in Ls:
        for trunc in truncations:
            for eps in eps_bits:
                H = build_H_fn(L=L, trunc=trunc, eps_bit=eps, **build_kwargs)
                evals = lowest_spectrum(H, solver=solver, k=k, **solver_kwargs)
                gap = float(evals[1] - evals[0]) if len(evals) >= 2 else float("nan")
                key = (L, trunc, eps)
                out[key] = dict(evals=evals, gap=gap)
                print(f"[CasRG] L={L} trunc={trunc} eps_bit={eps:g}  gap≈{gap:.6g}  evals[:4]={evals[:4]}")
    return out

# -----------------------------
# YOUR DATA HOOK (CasRG)
# -----------------------------
# Replace the placeholder below with YOUR actual model2 builder that returns a SciPy sparse matrix.
#
# def build_H_model2(L: int, trunc: Any, eps_bit: float, **kwargs):
#     ...
#     return Hs  # scipy.sparse.csr_matrix (symmetric)
#
# Then:
# results = casrg_gap_scan(build_H_model2,
#                          eps_bits=[0,1e-4,1e-3,1e-2],
#                          truncations=[J1,J2],
#                          Ls=[2,3],
#                          solver="lobpcg",
#                          solver_kwargs=dict(tol=1e-8, niter=250))


# ============================================================
# (4) SU(3) commutator seed swap: surrogate -> true ||[Aμ,Aν]||^2
# ============================================================

if jax is not None:
    # Use the standard SU(3) structure constants table:
    #   f_{123}=1
    #   f_{147}=f_{156}=f_{246}=f_{257}=f_{345}=f_{367}=1/2
    #   f_{458}=f_{678}=sqrt(3)/2
    def build_su3_f():
        f = np.zeros((8, 8, 8), float)
        entries = [
            (1, 2, 3, 1.0),
            (1, 4, 7, 0.5),
            (1, 5, 6, 0.5),
            (2, 4, 6, 0.5),
            (2, 5, 7, -0.5),
            (3, 4, 5, 0.5),
            (3, 6, 7, 0.5),
            (4, 5, 8, math.sqrt(3) / 2),
            (6, 7, 8, math.sqrt(3) / 2),
        ]
        for (a, b, c, val) in entries:
            a -= 1; b -= 1; c -= 1
            f[a, b, c] =  val
            f[b, c, a] =  val
            f[c, a, b] =  val
            f[a, c, b] = -val
            f[c, b, a] = -val
            f[b, a, c] = -val
        return jnp.array(f)

    f_su3 = build_su3_f()

    def su3_comm_seed_surrogate(theta_mu, kappa: float, f=f_su3):
        """
        Surrogate:
          kappa * Σ_{mu<nu} Σ_{b,c} (Σ_a f_{a b c}^2) (A_mu^b)^2 (A_nu^c)^2
        theta_mu: (...,4,8)
        """
        theta_mu = jnp.asarray(theta_mu)
        f2 = f * f
        M = jnp.sum(f2, axis=0)  # (8,8) on (b,c)
        pairs = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]
        tot = 0.0
        for mu, nu in pairs:
            A2 = theta_mu[..., mu, :] ** 2
            B2 = theta_mu[..., nu, :] ** 2
            tot = tot + jnp.einsum("...b,...c,bc->...", A2, B2, M)
        return kappa * tot

    def su3_comm_seed_true(theta_mu, kappa: float, f=f_su3):
        """
        True invariant:
          kappa * Σ_{mu<nu} ||[A_mu, A_nu]||^2
        with:
          [A_mu, A_nu]_a = Σ_{b,c} f_{a b c} A_mu^b A_nu^c
        theta_mu: (...,4,8)
        """
        theta_mu = jnp.asarray(theta_mu)
        pairs = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]
        tot = 0.0
        for mu, nu in pairs:
            A = theta_mu[..., mu, :]
            B = theta_mu[..., nu, :]
            comm = jnp.einsum("...b,...c,abc->...a", A, B, f)
            tot = tot + jnp.sum(comm * comm, axis=-1)
        return kappa * tot

    def _min_eig_hess(potential_fn, theta0):
        H = jax.hessian(lambda th: potential_fn(th))(theta0)
        H = np.array(H)
        evals = np.linalg.eigvalsh(H)
        return float(evals[0]), evals

    def demo_su3_comm_seed(kappa: float = 0.2, m2: float = 1.0, eps: float = 0.35, n_dirs: int = 10, seed: int = 0):
        rng = np.random.default_rng(seed)
        dim = 4 * 8

        direction = rng.normal(size=(dim,))
        direction = direction / np.linalg.norm(direction)

        def pot_true(theta_flat):
            th = theta_flat.reshape((4, 8))
            return 0.5 * m2 * jnp.sum(th * th) + su3_comm_seed_true(th, kappa=kappa)

        def pot_sur(theta_flat):
            th = theta_flat.reshape((4, 8))
            return 0.5 * m2 * jnp.sum(th * th) + su3_comm_seed_surrogate(th, kappa=kappa)

        theta_test = jnp.array(eps * direction)
        lam_min_true, evals_true = _min_eig_hess(pot_true, theta_test)
        lam_min_sur,  evals_sur  = _min_eig_hess(pot_sur,  theta_test)

        print("\n[DEMO] SU(3) commutator seed: surrogate vs true (Hessian + safe-radius proxy)")
        print(f"  At eps={eps:.3f}: min eig true={lam_min_true:.5f}, surrogate={lam_min_sur:.5f}")
        print(f"    true eig[0:6]={np.array(evals_true[:6])}")
        print(f"    sur  eig[0:6]={np.array(evals_sur[:6])}")

        def safe_radius_one(pot_fn, direction, eps_max=1.0, iters=30):
            lo, hi = 0.0, eps_max
            for _ in range(iters):
                mid = 0.5 * (lo + hi)
                lam_min, _ = _min_eig_hess(pot_fn, jnp.array(mid * direction))
                if (lam_min >= 0.0) and np.isfinite(lam_min):
                    lo = mid
                else:
                    hi = mid
            return lo

        dirs = rng.normal(size=(n_dirs, dim))
        dirs = dirs / np.linalg.norm(dirs, axis=1, keepdims=True)

        safe_true = np.array([safe_radius_one(pot_true, d) for d in dirs])
        safe_sur  = np.array([safe_radius_one(pot_sur,  d) for d in dirs])

        print(f"  grid-safe eps ({n_dirs} dirs): median true={np.median(safe_true):.3f}, surrogate={np.median(safe_sur):.3f}")
        print("  (Single-site proxy only; full lattice safe region will be smaller.)")
        return dict(lam_min_true=lam_min_true, lam_min_sur=lam_min_sur, safe_true=safe_true, safe_sur=safe_sur)

else:
    def demo_su3_comm_seed(*args, **kwargs):
        print("JAX not available; skipping SU(3) commutator seed demo.")
        return None


# ============================================================
# RUN DEMOS
# ============================================================
if __name__ == "__main__":
    demo_mass_module_synth_4d(seed=0)
    demo_su2_rg_intertwining(n_samples=4096, eps=0.1, iters=3, seed=0)
    demo_su3_comm_seed(kappa=0.2, m2=1.0, eps=0.35, n_dirs=10, seed=0)

    print("\n[DONE]")
    print(" - Mass module: use measure_mass_yukawa_prefactor(...) on your correlators.")
    print(" - SU(2) RG: batch_R_both(...) is keyword-safe static-arg JIT and reports intrinsic vs coord.")
    print(" - CasRG: swap lanczos_sparse_full_reorth(Hs) -> lowest_eigs_lobpcg(Hs,...) then scan with casrg_gap_scan.")
    print(" - SU(3): replace surrogate seed with su3_comm_seed_true(theta_mu,kappa) and compare Hessians/safe radii.")


[DEMO] Mass module on synthetic 4D Yukawa tail
  true m=0.550000   measured m=0.547899  (MAD~8.47e-04, window (29.5, 38.5))

[DEMO] SU(2) RG intertwining: intrinsic vs coordinate gradients
  intrinsic R: mean=1.00482 median=1.00474 max=1.01018
  coord     R: mean=0.250586 median=0.250588 max=0.252069
  coord/intrinsic median ratio ≈ 0.249405 (≈4 means quaternion-coordinate trap)

[DEMO] SU(3) commutator seed: surrogate vs true (Hessian + safe-radius proxy)
  At eps=0.350: min eig true=0.98878, surrogate=0.99611
    true eig[0:6]=[0.9887762  0.9892436  0.99339134 0.9953757  0.9961106  1.0006807 ]
    sur  eig[0:6]=[0.9961067 1.0008928 1.0035712 1.0052457 1.0067651 1.0074155]
  grid-safe eps (10 dirs): median true=1.000, surrogate=1.000
  (Single-site proxy only; full lattice safe region will be smaller.)

[DONE]
 - Mass module: use measure_mass_yukawa_prefactor(...) on your correlators.
 - SU(2) RG: batch_R_both(...) is keyword-safe static-arg JIT and reports intrinsic vs coord.
 - CasR

In [5]:
import torch
import numpy as np


# ============================================================
# 1) MASS MEASUREMENT MODULE
# ============================================================

def _to_torch(x, device="cuda" if torch.cuda.is_available() else "cpu", dtype=torch.float64):
    if isinstance(x, torch.Tensor):
        return x.to(device=device, dtype=dtype)
    return torch.as_tensor(x, device=device, dtype=dtype)

def tail_envelope_abs(vals: torch.Tensor) -> torch.Tensor:
    """env[r] = max_{s>=r} |vals[s]| (monotone tail envelope)."""
    v = torch.abs(vals)
    v_rev = torch.flip(v, dims=[0])
    env_rev = torch.cummax(v_rev, dim=0)[0]
    return torch.flip(env_rev, dims=[0])

def prefactor_correct(vals: torch.Tensor, power: float) -> torch.Tensor:
    """Multiply by (r+1)^power to remove asymptotic r^{-power} prefactor."""
    r = torch.arange(vals.numel(), device=vals.device, dtype=vals.dtype) + 1.0
    return (r ** power) * vals

def local_slopes_from_env(env: torch.Tensor, eps: float = 1e-300):
    """slopes[r] = -(log env[r+1] - log env[r]) with mids[r] = r+0.5."""
    y = torch.log(torch.clamp(env, min=eps))
    slopes = -(y[1:] - y[:-1])
    mids = torch.arange(0.5, 0.5 + slopes.numel(), device=env.device, dtype=env.dtype)
    return mids, slopes

def robust_plateau(slopes: torch.Tensor,
                   mids: torch.Tensor,
                   rmin: float,
                   rmax: float,
                   win: int = 8):
    """
    Pick the window of length win inside [rmin,rmax] with smallest MAD.
    Returns (m_hat, mad_hat, (r_left, r_right)).
    """
    mask = (mids >= rmin) & (mids <= rmax)
    s = slopes[mask]
    x = mids[mask]
    n = s.numel()
    if n < win:
        raise ValueError(f"Not enough slope points in [{rmin},{rmax}] (have {n}, need >= {win}).")

    best_m = None
    best_mad = None
    best_i = None
    for i in range(0, n - win + 1):
        w = s[i:i+win]
        med = torch.median(w)
        mad = torch.median(torch.abs(w - med))
        if (best_mad is None) or (mad < best_mad):
            best_mad = mad
            best_m = med
            best_i = i

    r_left  = float(x[best_i].item())
    r_right = float(x[best_i + win - 1].item())
    return float(best_m.item()), float(best_mad.item()), (r_left, r_right)

def torus_L1_dist_grid(L: int, d: int, device="cuda" if torch.cuda.is_available() else "cpu"):
    """dist[x] = sum_i min(x_i, L-x_i) on a d-dimensional L-torus."""
    coords = torch.meshgrid(*([torch.arange(L, device=device)] * d), indexing="ij")
    dist = torch.zeros([L] * d, device=device, dtype=torch.int32)
    for ax in range(d):
        x = coords[ax]
        dist += torch.minimum(x, (L - x))
    return dist

def env_by_dist_max(vals: torch.Tensor, dist: torch.Tensor, r_max: int):
    """env[r] = max_{x: dist[x]=r} vals[x]."""
    v = vals.reshape(-1)
    d = dist.reshape(-1).to(torch.int64)
    env = torch.zeros((r_max + 1,), device=vals.device, dtype=vals.dtype)
    return env.scatter_reduce(0, d, v, reduce="amax", include_self=True)

def directional_profile_abs(G: torch.Tensor,
                            direction,
                            L: int,
                            d: int,
                            rmax: int):
    """
    Sample |G| along ray x(r)=r*direction mod L, r=0..rmax.
    Returns vals[r] and step_norm=||direction||_2 (so you can convert "per-step" to "per-euclidean").
    """
    dirv = torch.tensor(direction, device=G.device, dtype=torch.int64)
    if dirv.numel() != d:
        raise ValueError(f"direction must have length d={d}, got {dirv.numel()}")
    step_norm = float(torch.sqrt(torch.sum(dirv.to(torch.float64)**2)).item())
    pos = torch.zeros((d,), device=G.device, dtype=torch.int64)
    vals = []
    for _ in range(rmax + 1):
        vals.append(torch.abs(G[tuple(pos.tolist())]))
        pos = (pos + dirv) % L
    return torch.stack(vals, dim=0), step_norm

class MassMeasurer:
    """
    Mass estimate from correlators by:
      - take |C|
      - prefactor-correct (multiply by (r+1)^{(d-1)/2} by default)
      - take monotone tail envelope
      - take log-slopes
      - pick a stable plateau window (min MAD)
    """

    @staticmethod
    def from_1d(vals,
                d: int,
                rmin: float,
                rmax: float,
                prefactor_power: float = None,
                win: int = 8):
        vals = _to_torch(vals)
        if prefactor_power is None:
            prefactor_power = 0.5 * (d - 1)

        corrected = prefactor_correct(torch.abs(vals), prefactor_power)
        env = tail_envelope_abs(corrected)
        mids, slopes = local_slopes_from_env(env)
        m_hat, mad, rng = robust_plateau(slopes, mids, rmin=rmin, rmax=rmax, win=win)
        return dict(m_hat=m_hat, mad=mad, plateau_mids=rng, env=env.detach().cpu(), mids=mids.detach().cpu(), slopes=slopes.detach().cpu())

    @staticmethod
    def from_torus_shell(Gx,
                         L: int,
                         d: int,
                         rmin: float,
                         rmax: float,
                         prefactor_power: float = None,
                         win: int = 8):
        """
        Shell version: env[r]=max_{L1(x)=r} |G(x)|.
        """
        Gx = _to_torch(Gx)
        if prefactor_power is None:
            prefactor_power = 0.5 * (d - 1)

        dist = torus_L1_dist_grid(L, d, device=Gx.device)
        r_max_total = int(d * (L // 2))
        env = env_by_dist_max(torch.abs(Gx), dist, r_max_total)

        corrected_env = prefactor_correct(env, prefactor_power)
        env2 = tail_envelope_abs(corrected_env)
        mids, slopes = local_slopes_from_env(env2)
        m_hat, mad, rng = robust_plateau(slopes, mids, rmin=rmin, rmax=rmax, win=win)
        return dict(m_hat=m_hat, mad=mad, plateau_mids=rng, env=env.detach().cpu(), mids=mids.detach().cpu(), slopes=slopes.detach().cpu())

    @staticmethod
    def from_torus_directions(Gx,
                              L: int,
                              d: int,
                              directions,
                              rmin: float,
                              rmax: float,
                              rmax_dir: int = None,
                              prefactor_power: float = None,
                              win: int = 8,
                              convert_to_euclidean: bool = True):
        """
        Directional version: sample along a few rays and combine.
        """
        Gx = _to_torch(Gx)
        if prefactor_power is None:
            prefactor_power = 0.5 * (d - 1)
        if rmax_dir is None:
            rmax_dir = max(1, L//2 - 2)

        per = []
        for direction in directions:
            vals, step_norm = directional_profile_abs(Gx, direction, L=L, d=d, rmax=rmax_dir)
            res = MassMeasurer.from_1d(vals, d=d, rmin=rmin, rmax=rmax, prefactor_power=prefactor_power, win=win)
            m_step = res["m_hat"]
            m_eucl = m_step / step_norm if convert_to_euclidean and step_norm > 0 else m_step
            per.append((tuple(direction), step_norm, m_eucl, m_step, res))

        masses = torch.tensor([x[2] for x in per], dtype=torch.float64)
        m_med = float(torch.median(masses).item())
        mad = float(torch.median(torch.abs(masses - torch.median(masses))).item())
        return dict(m_median=m_med, mad=mad, per_direction=per)

    @staticmethod
    def cosh_meff(Ct, tmin: int = 2, tmax: int = None):
        """
        Periodic-time effective mass:
          m_eff(t) = arcosh( (C(t-1)+C(t+1))/(2C(t)) )
        """
        Ct = _to_torch(Ct, device="cpu")
        T = Ct.numel()
        if tmax is None:
            tmax = T - 2
        ts, meff = [], []
        for t in range(tmin, tmax + 1):
            num = Ct[(t-1) % T] + Ct[(t+1) % T]
            den = 2.0 * Ct[t % T]
            x = float((num/den).clamp(min=1.0 + 1e-12).item())
            ts.append(t)
            meff.append(math.acosh(x))
        return np.array(ts), np.array(meff)

# --- 1. Generate Synthetic 4D Torus Correlator Data ---
# Let's create a synthetic correlator Gx on a 4D torus of size L.
# We'll make it decay exponentially with the L1 distance from the origin.

L = 16  # Lattice size
d = 4   # Dimensions
m_true = 0.8 # True mass for our synthetic data

# Create a distance grid for L1 norm on a torus
def torus_L1_dist_grid_np(L: int, d: int):
    coords = np.meshgrid(*([np.arange(L)] * d), indexing="ij")
    dist = np.zeros([L] * d, dtype=np.int32)
    for ax in range(d):
        x = coords[ax]
        dist += np.minimum(x, (L - x))
    return dist

dist_grid = torus_L1_dist_grid_np(L, d)

# Generate the correlator values based on distance and true mass
# Add some noise to make it realistic
np.random.seed(42)
G_clean = np.exp(-m_true * dist_grid)
noise_level = 0.05
G_noisy = G_clean * (1 + noise_level * np.random.randn(*G_clean.shape))

# Convert to torch tensor
Gx_synthetic = torch.tensor(G_noisy, dtype=torch.float64)

print(f"Generated synthetic {d}D torus correlator data of shape {Gx_synthetic.shape}")

# --- 2. Use MassMeasurer.from_torus_shell to measure mass ---

# Define the analysis range for mass measurement
rmin_analysis = 2.0  # Minimum L1 distance to consider for plateau
rmax_analysis = float(L // 2 - 2) # Maximum L1 distance, avoiding wrap-around effects

print(f"\nAttempting mass measurement with rmin={rmin_analysis}, rmax={rmax_analysis}")

# Call the mass measurer
result_shell = MassMeasurer.from_torus_shell(
    Gx=Gx_synthetic,
    L=L,
    d=d,
    rmin=rmin_analysis,
    rmax=rmax_analysis,
    win=4 # Window size for robust plateau finding, adjusted from 8 to 4
)

# --- 3. Print the results ---
print("\nMass Measurement Results (from_torus_shell):")
print(f"  True Mass: {m_true:.6f}")
print(f"  Measured Mass (m_hat): {result_shell['m_hat']:.6f}")
print(f"  MAD of Plateau: {result_shell['mad']:.2e}")
print(f"  Plateau window (r_mid units): {result_shell['plateau_mids']}")


Generated synthetic 4D torus correlator data of shape torch.Size([16, 16, 16, 16])

Attempting mass measurement with rmin=2.0, rmax=6.0


ValueError: Not enough slope points in [2.0,6.0] (have 4, need >= 8).

In [4]:
# ============================================================
# ONE-CELL RUN BLOCK (Colab-friendly)
#   1) Prefactor-corrected plateau extractor => Mass measurement module
#   2) SU(2) RG-intertwining constant test (JAX static-arg fix + intrinsic tangent norms)
#   3) CasRG model2: swap Lanczos(full reorth) -> LOBPCG / IRL / shift-invert
#   4) SU(3) commutator seed: surrogate -> true ||[A_mu,A_nu]||^2 + Hessian/safe-radius compare
# ============================================================

import math
import numpy as np
import torch

TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE  = torch.float64
torch.set_default_dtype(TORCH_DTYPE)


# ============================================================
# 2) SU(2) RG-INTERTWINING TEST (JAX)
# ============================================================

try:
    import jax
    import jax.numpy as jnp
    from functools import partial
except Exception as e:
    jax = None
    print("\n[WARN] JAX+JAXLIB not available => SU(2)/SU(3) parts will be skipped.\n", e)

if jax is not None:
    # ---- SU(2) quaternion ops
    def quat_mul(q1, q2):
        w1,x1,y1,z1 = jnp.split(q1, 4, axis=-1)
        w2,x2,y2,z2 = jnp.split(q2, 4, axis=-1)
        return jnp.concatenate([
            w1*w2 - x1*x2 - y1*y2 - z1*z2,
            w1*x2 + x1*w2 + y1*z2 - z1*y2,
            w1*y2 - x1*z2 + y1*w2 + z1*x2,
            w1*z2 + x1*y2 - y1*x2 + z1*w2
        ], axis=-1)

    def quat_inv(q):
        return jnp.concatenate([q[..., :1], -q[..., 1:]], axis=-1)

    def quat_unit(q):
        return q / (jnp.sqrt(jnp.sum(q*q, axis=-1, keepdims=True)) + 1e-12)

    def su2_exp(x):
        theta = jnp.linalg.norm(x, axis=-1, keepdims=True)
        half = 0.5 * theta
        w = jnp.cos(half)
        s = jnp.where(theta > 1e-12, jnp.sin(half)/theta, 0.5 - (theta*theta)/48.0)
        xyz = s * x
        return jnp.concatenate([w, xyz], axis=-1)

    def su2_log(q):
        q = quat_unit(q)
        w = q[..., :1]
        v = q[..., 1:]
        nv = jnp.linalg.norm(v, axis=-1, keepdims=True)
        ang = 2.0 * jnp.arctan2(nv, w)
        coef = jnp.where(nv > 1e-12, ang/nv, 2.0/(w + 1e-12))
        return coef * v

    def karcher_mean(quats, iters=5):
        q0 = quat_unit(quats[0])
        def body(_, qcur):
            logs = su2_log(quat_mul(quat_inv(qcur), quats))
            delta = jnp.mean(logs, axis=0)
            return quat_unit(quat_mul(qcur, su2_exp(delta)))
        return jax.lax.fori_loop(0, iters, body, q0)

    def pi_geodesic(U_block, iters=5):
        return su2_log(karcher_mean(U_block, iters=iters))  # R^3

    def compute_R_coord(U_block, v, iters=5):
        # coordinate gradient in raw quaternion coords (factor-of-4 trap)
        def F_pullback(flatU):
            U = flatU.reshape((-1,4))
            Y = pi_geodesic(U, iters=iters)
            return jnp.dot(v, Y)
        g = jax.grad(F_pullback)(U_block.reshape(-1)).reshape((-1,4))[:,1:]
        return jnp.sum(g*g) / (jnp.sum(v*v) + 1e-12)

    def compute_R_intrinsic(U_block, v, iters=5):
        # intrinsic gradient in Lie algebra coords (no factor-of-4 trap)
        n_links = U_block.shape[0]
        X0 = jnp.zeros((n_links,3), dtype=U_block.dtype)
        def F_pullback(X):
            U_pert = quat_mul(U_block, su2_exp(X))  # right-multiply by exp(X)
            Y = pi_geodesic(U_pert, iters=iters)
            return jnp.dot(v, Y)
        gX = jax.grad(F_pullback)(X0)
        return jnp.sum(gX*gX) / (jnp.sum(v*v) + 1e-12)

    @partial(jax.jit, static_argnames=("n_samples","n_links","iters"))
    def batch_R(key, n_samples: int = 512, n_links: int = 16, eps: float = 0.15, iters: int = 5):
        """
        FIXED: static_argnames (keyword-safe) so n_samples can safely control shapes.
        """
        keys = jax.random.split(key, n_samples)
        def one(k):
            k1, k2 = jax.random.split(k)
            X = eps * jax.random.normal(k1, (n_links, 3))
            U = su2_exp(X)
            v = jax.random.normal(k2, (3,))
            v = v / (jnp.linalg.norm(v) + 1e-12)
            return compute_R_intrinsic(U, v, iters=iters), compute_R_coord(U, v, iters=iters)
        Rin, Rco = jax.vmap(one)(keys)
        return Rin, Rco

    def run_rg_test():
        print("\n[DEMO] SU(2) RG intertwining: intrinsic vs coordinate gradients")
        Rin, Rco = batch_R(jax.random.PRNGKey(0), n_samples=256, n_links=16, eps=0.15, iters=5)
        Rin = np.array(Rin); Rco = np.array(Rco)
        print(f"  intrinsic R: mean={Rin.mean():.6g} median={np.median(Rin):.6g} max={Rin.max():.6g}")
        print(f"  coord     R: mean={Rco.mean():.6g} median={np.median(Rco):.6g} max={Rco.max():.6g}")
        print(f"  coord/intrinsic median ratio \u2248 {np.median(Rco)/np.median(Rin):.3g} (\u22484 means you hit the quaternion-coordinate trap)")

# ============================================================
# 3) CASRG MODEL2: cheaper eigensolvers (LOBPCG / IRL / shift-invert)
# ============================================================

def symmetrize_sparse(H: torch.Tensor) -> torch.Tensor:
    """(H + H^T)/2 for sparse COO."""
    Hs = (H + H.transpose(0,1)) * 0.5
    return Hs.coalesce()

def lowest_eigs_lobpcg(H: torch.Tensor, k: int = 4, tol: float = 1e-8, niter: int = 250, seed: int = 0):
    """
    Compute k smallest eigenvalues with torch.lobpcg (GPU-friendly).
    H must be symmetric and sparse.
    """
    H = symmetrize_sparse(H)
    n = H.shape[0]
    gen = torch.Generator(device=H.device)
    gen.manual_seed(seed)
    X = torch.randn((n,k), device=H.device, dtype=H.dtype, generator=gen)
    X, _ = torch.linalg.qr(X, mode="reduced")
    evals, _ = torch.lobpcg(H, k=k, largest=False, tol=tol, niter=niter, init=X)
    evals = evals.detach().cpu().numpy()
    evals.sort()
    return evals

def torch_sparse_to_scipy_csr(H: torch.Tensor):
    import scipy.sparse as sp
    H = H.coalesce().cpu()
    idx = H.indices().numpy()
    dat = H.values().numpy()
    return sp.coo_matrix((dat, (idx[0], idx[1])), shape=H.shape).tocsr()

def lowest_eigs_eigsh(H: torch.Tensor, k: int = 4, which: str = "SA", tol: float = 1e-10, maxiter: int = 5000):
    """
    CPU IRL (scipy.sparse.linalg.eigsh). Reliable, but moves H to CPU.
    """
    import scipy.sparse.linalg as spla
    Hcsr = torch_sparse_to_scipy_csr(H)
    evals, _ = spla.eigsh(Hcsr, k=k, which=which, tol=tol, maxiter=maxiter)
    evals = np.array(evals, dtype=float)
    evals.sort()
    return evals

def lowest_eigs_shift_invert(H: torch.Tensor, k: int = 4, sigma: float = 0.0, tol: float = 1e-10, maxiter: int = 5000):
    """
    CPU shift-invert around sigma using scipy.eigsh(..., sigma=sigma).
    Good if you need very accurate low-lying spectrum and H is ill-conditioned.
    """
    import scipy.sparse.linalg as spla
    Hcsr = torch_sparse_to_scipy_csr(H)
    evals, _ = spla.eigsh(Hcsr, k=k, sigma=sigma, which="LM", tol=tol, maxiter=maxiter)
    evals = np.array(evals, dtype=float)
    evals.sort()
    return evals

def gap_from_eigs(evals):
    evals = np.array(evals, dtype=float)
    evals.sort()
    return float(evals[0]), float(evals[1] - evals[0])

def casrg_gap_scan(build_H_fn,
                   eps_bits,
                   truncations,
                   Ls,
                   solver: str = "lobpcg",
                   eig_k: int = 4):
    """
    build_H_fn(L, trunc, eps_bit) -> torch sparse COO (symmetric).
    Returns list of dicts with (L,trunc,eps_bit,e0,gap,evals).
    """
    out = []
    for L in Ls:
        for trunc in truncations:
            for eps in eps_bits:
                H = build_H_fn(L=L, trunc=trunc, eps_bit=eps).to(device=TORCH_DEVICE, dtype=TORCH_DTYPE)
                if solver == "lobpcg":
                    evals = lowest_eigs_lobpcg(H, k=eig_k)
                elif solver == "eigsh":
                    evals = lowest_eigs_eigsh(H, k=eig_k)
                elif solver == "shift-invert":
                    evals = lowest_eigs_shift_invert(H, k=eig_k, sigma=0.0)
                else:
                    raise ValueError("solver must be 'lobpcg', 'eigsh', or 'shift-invert'")
                e0, gap = gap_from_eigs(evals)
                out.append(dict(L=L, trunc=trunc, eps_bit=eps, e0=e0, gap=gap, evals=evals))
                print(f"[CasRG] L={L} trunc={trunc} eps_bit={eps:.3g}  e0={e0:.6g} gap={gap:.6g}")
    return out

def print_casrg_patch_notes():
    print(r"""
[PATCH NOTES] CasRG model2 eigen solver swap

Where you currently run full-reorth Lanczos (slow):
    evals = lanczos_sparse_full_reorth(Hs)

Swap to:
    evals = lowest_eigs_lobpcg(Hs, k=4, tol=1e-8, niter=250)

Then do a modest scan like:
    results = casrg_gap_scan(build_H_model2, eps_bits=[0,1e-4,1e-3,1e-2],
                             truncations=[J1,J2], Ls=[2,3], solver="lobpcg")

If the lowest spectrum is *very* clustered / ill-conditioned, use IRL or shift-invert:
    solver="eigsh"     (robust)
    solver="shift-invert"  (most accurate, CPU + factorization)
""")

# ============================================================
# 4) SU(3) COMMUTATOR SEED UPGRADE (JAX)
# ============================================================

if jax is not None:
    def build_su3_f_tensor():
        f = np.zeros((8,8,8), dtype=float)
        entries = [
            (1,2,3, 1.0),
            (1,4,7, 0.5),
            (1,5,6, 0.5),
            (2,4,6, 0.5),
            (2,5,7,-0.5),
            (3,4,5, 0.5),
            (3,6,7, 0.5),
            (4,5,8, math.sqrt(3)/2),
            (6,7,8, math.sqrt(3)/2),
        ]
        for (a,b,c,val) in entries:
            a-=1; b-=1; c-=1
            f[a,b,c] =  val
            f[b,c,a] =  val
            f[c,a,b] =  val
            f[a,c,b] = -val
            f[c,b,a] = -val
            f[b,a,c] = -val
        return jnp.array(f)

    F_SU3 = build_su3_f_tensor()
    M_SU3 = jnp.sum(F_SU3 * F_SU3, axis=0)  # M_bc = \u03A3_a f[a,b,c]^2

    def su3_comm_true_pair(A, B):
        comm = jnp.einsum("abc,...b,...c->...a", F_SU3, A, B)
        return jnp.sum(comm*comm, axis=-1)

    def su3_comm_surrogate_pair(A, B):
        A2 = A*A
        B2 = B*B
        return jnp.einsum("...b,...c,bc->...", A2, B2, M_SU3)

    def su3_comm_seed_true(theta_mu, kappa: float):
        """
        TRUE: kappa * \u03A3_{mu<nu} ||[A_mu, A_nu]||^2
        theta_mu: (...,4,8)
        """
        total = 0.0
        for mu in range(4):
            for nu in range(mu+1, 4):
                total = total + su3_comm_true_pair(theta_mu[...,mu,:], theta_mu[...,nu,:])
        return kappa * total

    def su3_comm_seed_surrogate(theta_mu, kappa: float):
        """
        SURROGATE: kappa * \u03A3_{mu<nu} \u03A3_{b,c} M_bc (A_mu_b^2)(A_nu_c^2)
        theta_mu: (...,4,8)
        """
        total = 0.0
        for mu in range(4):
            for nu in range(mu+1, 4):
                total = total + su3_comm_surrogate_pair(theta_mu[...,mu,:], theta_mu[...,nu,:])
        return kappa * total

    # ---- tiny Wilson single-site + Hessian compare (proxy for your full convexity engine)
    def gell_mann():
        i    = 0.0 + 1.0j
        lam = []
        lam.append(jnp.array([[0,1,0],[1,0,0],[0,0,0]], dtype=jnp.complex64))
        lam.append(jnp.array([[0,-i,0],[i,0,0],[0,0,0]], dtype=jnp.complex64))
        lam.append(jnp.array([[1,0,0],[0,-1,0],[0,0,0]], dtype=jnp.complex64))
        lam.append(jnp.array([[0,0,1],[0,0,0],[1,0,0]], dtype=jnp.complex64))
        lam.append(jnp.array([[0,0,-i],[0,0,0],[i,0,0]], dtype=jnp.complex64))
        lam.append(jnp.array([[0,0,0],[0,0,1],[0,1,0]], dtype=jnp.complex64))
        lam.append(jnp.array([[0,0,0],[0,0,-i],[0,i,0]], dtype=jnp.complex64))
        lam.append((1.0/math.sqrt(3.0))*jnp.array([[1,0,0],[0,1,0],[0,0,-2]], dtype=jnp.complex64))
        return jnp.stack(lam, axis=0)

    LAM = gell_mann()
    T_BASIS = (0.5j) * LAM  # anti-Hermitian basis

    def su3_alg_from_vec(theta):
        return jnp.einsum("...a,aij->...ij", theta, T_BASIS)

    def su3_exp(theta):
        import jax.scipy.linalg as jsp
        return jsp.expm(su3_alg_from_vec(theta))

    def su3_dag(U):
        return jnp.swapaxes(jnp.conjugate(U), -1, -2)

    def wilson_single_site(theta_mu, beta: float):
        U = jax.vmap(su3_exp)(theta_mu)  # (4,3,3)
        S = 0.0
        for mu in range(4):
            for nu in range(mu+1, 4):
                P = U[mu] @ U[nu] @ su3_dag(U[mu]) @ su3_dag(U[nu])
                S = S + (1.0 - jnp.real(jnp.trace(P))/3.0)
        return beta * S

    def action_seeded(theta_flat, beta: float, kappa: float, which: str):
        theta_mu = theta_flat.reshape((4,8))
        m2 = 0.2
        base = wilson_single_site(theta_mu, beta=beta) + 0.5*m2*jnp.sum(theta_mu*theta_mu)
        if which == "true":
            return base + su3_comm_seed_true(theta_mu, kappa=kappa)
        if which == "surrogate":
            return base + su3_comm_seed_surrogate(theta_mu, kappa=kappa)
        raise ValueError

    def hessian_spectrum(action_fn, x):
        H = jax.hessian(action_fn)(x)
        evals = jnp.linalg.eigvalsh(H)
        return evals

    def safe_radius_grid(action_fn, dim: int, key, eps_grid, n_dirs: int = 12):
        keys = jax.random.split(key, n_dirs)
        def one(k):
            u = jax.random.normal(k, (dim,))
            u = u / (jnp.linalg.norm(u) + 1e-12)
            safe = 0.0
            for eps in eps_grid:
                evals = hessian_spectrum(action_fn, eps*u)
                safe = jnp.where(evals[0] > 0.0, eps, safe)
            return safe
        return np.array(jax.vmap(one)(keys))

    def demo_su3_seed():
        print("\n[DEMO] SU(3) commutator seed: surrogate vs true (Hessian + safe-radius proxy)")
        beta = 0.20
        kappa = 0.50
        dim = 32

        a_true = lambda z: action_seeded(z, beta=beta, kappa=kappa, which="true")
        a_sur  = lambda z: action_seeded(z, beta=beta, kappa=kappa, which="surrogate")

        key = jax.random.PRNGKey(123)
        u = jax.random.normal(key, (dim,))
        u = u / (jnp.linalg.norm(u) + 1e-12)
        eps = 0.35

        evals_true = np.array(hessian_spectrum(a_true, eps*u))
        evals_sur  = np.array(hessian_spectrum(a_sur,  eps*u))
        print(f"  At eps={eps:.3f}: min eig true={evals_true[0]:.6g}, surrogate={evals_sur[0]:.6g}")
        print(f"    true eig[0:6]={evals_true[:6]}")
        print(f"    sur  eig[0:6]={evals_sur[:6]}")

        eps_grid = np.linspace(0.0, 0.8, 21)
        safe_true = safe_radius_grid(a_true, dim, jax.random.PRNGKey(1), eps_grid, n_dirs=10)
        safe_sur  = safe_radius_grid(a_sur,  dim, jax.random.PRNGKey(2), eps_grid, n_dirs=10)
        print(f"  grid-safe eps (10 dirs): median true={np.median(safe_true):.3f}, surrogate={np.median(safe_sur):.3f}")
        print("  (This is a single-site proxy; your full lattice safe region will be smaller.)")

    def print_su3_swap_notes():
        print(r"""
[PATCH NOTES] SU(3) commutator seed swap

Where you currently use a 'square-of-squares' surrogate, replace it with:

    seed = su3_comm_seed_true(theta_mu, kappa)

with theta_mu shaped (...,4,8) (four directions, adjoint coords).

That computes:
    kappa * \u03A3_{mu<nu} ||[A_mu, A_nu]||^2
with commutator components:
    [A_mu, A_nu]_a = \u03A3_{b,c} f_{a b c} A_mu^b A_nu^c

Then re-run your Hessian-spectrum + safe-region scan to compare against:
    su3_comm_seed_surrogate(theta_mu, kappa)
""")

# ============================================================
# RUN (set these flags)
# ============================================================

RUN_MASS_DEMO = False # Set to False since the demo logic is moved
RUN_RG_DEMO   = True
RUN_SU3_DEMO  = True

if RUN_MASS_DEMO:
    _demo_mass()

print_casrg_patch_notes()

if jax is not None and RUN_RG_DEMO:
    run_rg_test()

if jax is not None:
    print_su3_swap_notes()
    if RUN_SU3_DEMO:
        demo_su3_seed()

# ============================================================
# HOW TO USE ON YOUR DATA (quick recipes)
# ============================================================
#
# (A) Interacting correlator on T^4 (gauge-projected kernel, etc):
#     Gx = ...  # numpy/torch array shape [L,L,L,L], translation invariant
#     res = MassMeasurer.from_torus_shell(Gx, L=L, d=4, rmin=10, rmax=30)
#     print(res["m_hat"])
#
# (B) Directional masses:
#     dirs = [(1,0,0,0),(1,1,0,0),(1,1,1,0),(1,1,1,1)]
#     res = MassMeasurer.from_torus_directions(Gx, L=L, d=4, directions=dirs,
#                                              rmin=6, rmax=20, rmax_dir=L//2-2)
#     print(res["m_median"], res["mad"])
#
# (C) Wilson-loop / temporal correlator C(t) (periodic):
#     ts, meff = MassMeasurer.cosh_meff(Ct)
#     # then plateau-pick on meff if you like, or do tail-envelope slopes with prefactor_power=0:
#     out = MassMeasurer.from_1d(Ct, d=1, rmin=4, rmax=12, prefactor_power=0.0)
#     print("V(R)~", out["m_hat"])
#
# (D) CasRG model2 gap scan:
#     def build_H_model2(L, trunc, eps_bit): ...
#     results = casrg_gap_scan(build_H_model2,
#                              eps_bits=[0.0,1e-4,1e-3,1e-2,1e-1],
#                              truncations=[...],
#                              Ls=[2,3],
#                              solver="lobpcg")
# ============================================================



[PATCH NOTES] CasRG model2 eigen solver swap

Where you currently run full-reorth Lanczos (slow):
    evals = lanczos_sparse_full_reorth(Hs)

Swap to:
    evals = lowest_eigs_lobpcg(Hs, k=4, tol=1e-8, niter=250)

Then do a modest scan like:
    results = casrg_gap_scan(build_H_model2, eps_bits=[0,1e-4,1e-3,1e-2],
                             truncations=[J1,J2], Ls=[2,3], solver="lobpcg")

If the lowest spectrum is *very* clustered / ill-conditioned, use IRL or shift-invert:
    solver="eigsh"     (robust)
    solver="shift-invert"  (most accurate, CPU + factorization)


[DEMO] SU(2) RG intertwining: intrinsic vs coordinate gradients
  intrinsic R: mean=nan median=nan max=nan
  coord     R: mean=nan median=nan max=nan
  coord/intrinsic median ratio ≈ nan (≈4 means you hit the quaternion-coordinate trap)

[PATCH NOTES] SU(3) commutator seed swap

Where you currently use a 'square-of-squares' surrogate, replace it with:

    seed = su3_comm_seed_true(theta_mu, kappa)

with theta_mu shap

/usr/local/lib/python3.12/dist-packages/jax/_src/lax/lax.py:5473: ComplexWarning: Casting complex values to real discards the imaginary part
  x_bar = _convert_element_type(x_bar, x.aval.dtype, x.aval.weak_type)


  At eps=0.350: min eig true=0.17211, surrogate=0.19841
    true eig[0:6]=[0.17210972 0.17565002 0.18067285 0.18342575 0.19575062 0.20200203]
    sur  eig[0:6]=[0.19841032 0.20302065 0.20454891 0.20588928 0.21531667 0.21687661]
  grid-safe eps (10 dirs): median true=0.800, surrogate=0.800
  (This is a single-site proxy; your full lattice safe region will be smaller.)


In [6]:
# ============================================================
# MASTER PATCH CELL (Dec-2025): Mass module + SU(2) RG constant (intrinsic-fixed)
#                               + CasRG(model2) LOBPCG swap + SU(3) commutator seed (true invariant)
# ============================================================

# -----------------------------
# USER KNOBS (change these!)
# -----------------------------
RUN_MASS_DEMO      = True
RUN_RG_DEMO        = True
RUN_SU3_DEMO       = True
RUN_CASRG_MODEL2   = False   # heavy (basis build + eigensolves). Flip True when you're ready.

# Mass extractor knobs
MASS_DIM           = 4
MASS_RMIN          = 10.0     # push larger to avoid pre-asymptotic region
MASS_RMAX          = None     # None -> use max available
MASS_WMIN          = 8
MASS_WMAX          = 20

# SU(2) RG intertwining knobs
RG_NLINKS          = 16       # 2x2x2x2 block
RG_ITERS           = 3        # Karcher mean iterations
RG_EPS             = 0.05     # small => linear regime
RG_NSAMPLES        = 2048     # increase for tighter stats (4096/8192 ok)

# SU(3) commutator seed knobs
SU3_KAPPA          = 0.30
SU3_EPS0           = 0.35     # point where we print Hessian spectra
SU3_NDIRS          = 12
SU3_EPS_GRID       = [0.0, 0.2, 0.35, 0.5, 0.7, 0.9, 1.1, 1.3, 1.5]

# CasRG(model2) knobs (matches your one-shot driver structure)
CASRG_L            = 2
CASRG_ALPHA_LIST   = [0.5, 1.0]
CASRG_BETA_LIST    = [0.3, 1.0, 3.0]
CASRG_EPS_BITS     = [0.0, 1e-4, 1e-3, 1e-2, 1e-1, 0.2]  # small-to-moderate scan
CASRG_SOLVER       = "lobpcg"  # "lobpcg" (fast), "eigsh" (robust CPU), "shift-invert" (very accurate CPU)
CASRG_K_EIGS       = 6
CASRG_TOL          = 1e-8
CASRG_NITER        = 250
CASRG_SEED         = 1

# ============================================================
# 0) Imports
# ============================================================
import math, itertools, csv
from collections import defaultdict
import numpy as np

import torch

# ============================================================
# 1) MASS MEASUREMENT MODULE (prefactor-corrected Yukawa tail)
# ============================================================
def _tail_envelope_torch(y: torch.Tensor) -> torch.Tensor:
    """Monotone tail envelope: env[r] = max_{s>=r} y[s]."""
    env = y.clone()
    for i in range(env.numel() - 2, -1, -1):
        env[i] = torch.maximum(env[i], env[i+1])
    return env

def _prefactor_correct_torch(y: torch.Tensor, d: int) -> torch.Tensor:
    """Multiply by (r+1)^{(d-1)/2} to flatten Yukawa prefactor in d dims."""
    r = torch.arange(y.numel(), device=y.device, dtype=y.dtype)
    power = 0.5 * (d - 1)
    return y * (r + 1.0) ** power

def _local_slopes_torch(env: torch.Tensor, step_len: float, eps: float = 1e-300):
    """Slope of -log(env) per unit *Euclidean* step length."""
    env_clamped = torch.clamp(env, min=eps)
    logy = torch.log(env_clamped)
    slopes = -(logy[1:] - logy[:-1]) / step_len
    mids = (torch.arange(slopes.numel(), device=env.device, dtype=env.dtype) + 0.5)
    return mids, slopes

def _plateau_search_torch(mids: torch.Tensor,
                          slopes: torch.Tensor,
                          rmin: float,
                          rmax: float,
                          wmin: int,
                          wmax: int):
    """Pick window minimizing MAD (robust). Returns (median, mad, (r_lo,r_hi))."""
    if slopes.numel() < wmin:
        med = float(torch.median(slopes).item())
        mad = float(torch.median(torch.abs(slopes - torch.tensor(med, device=slopes.device, dtype=slopes.dtype))).item())
        return med, mad, (float(mids[0].item()), float(mids[-1].item()))

    mask = (mids >= rmin) & (mids <= rmax)
    idx = torch.nonzero(mask).squeeze(-1)
    if idx.numel() == 0:
        # fall back: use full range
        idx0, idx1 = 0, slopes.numel()-1
    else:
        idx0, idx1 = int(idx[0].item()), int(idx[-1].item())

    best = None
    max_w = min(wmax, idx1 - idx0 + 1)
    for w in range(wmin, max_w + 1):
        for i in range(idx0, idx1 - w + 2):
            win = slopes[i:i+w]
            med = torch.median(win)
            mad = torch.median(torch.abs(win - med))
            score = float(mad.item())
            if (best is None) or (score < best["score"]):
                best = dict(i=i, w=w, med=med, mad=mad, score=score)

    i, w = best["i"], best["w"]
    r_lo = float(mids[i].item())
    r_hi = float(mids[i+w-1].item())
    return float(best["med"].item()), float(best["mad"].item()), (r_lo, r_hi)

def measure_mass_from_profile(profile: torch.Tensor,
                              d: int = 4,
                              step_len: float = 1.0,
                              rmin: float = 8.0,
                              rmax: float | None = None,
                              wmin: int = 8,
                              wmax: int = 20,
                              use_prefactor: bool = True):
    """
    profile[r] ~ exp(-m r)/r^{(d-1)/2} (up to lattice/torus artifacts).
    Returns dict with m, MAD, and chosen plateau window.
    """
    y = torch.abs(profile)
    if use_prefactor:
        y = _prefactor_correct_torch(y, d=d)
    env = _tail_envelope_torch(y)

    mids, slopes = _local_slopes_torch(env, step_len=step_len)
    if rmax is None:
        rmax = float(mids[-1].item())

    m_hat, mad, window = _plateau_search_torch(mids, slopes, rmin=rmin, rmax=rmax, wmin=wmin, wmax=wmax)
    return dict(m=m_hat, mad=mad, window=window, step_len=step_len)

def ray_samples_abs(G: torch.Tensor, direction: tuple[int, ...], n_steps: int | None = None, origin: tuple[int, ...] | None = None):
    """
    Sample |G(x)| along a ray on a periodic lattice starting at origin, stepping by 'direction'.
    """
    d = G.ndim
    assert len(direction) == d
    Ls = list(G.shape)

    if origin is None:
        origin = tuple(0 for _ in range(d))
    idx = list(origin)

    if n_steps is None:
        # Default: stop before obvious wrap-around; conservative.
        n_steps = min((Ls[a] // 2) if direction[a] != 0 else 10**9 for a in range(d)) + 1

    out = []
    for _ in range(n_steps):
        out.append(torch.abs(G[tuple(idx)]))
        for a, step in enumerate(direction):
            idx[a] = (idx[a] + step) % Ls[a]
    return torch.stack(out)

def measure_mass_yukawa_prefactor(G: torch.Tensor,
                                 direction: tuple[int, ...],
                                 d: int = 4,
                                 rmin: float = 8.0,
                                 rmax: float | None = None,
                                 wmin: int = 8,
                                 wmax: int = 20):
    """
    Mass from lattice correlator G[...] using direction ray + prefactor correction.
    Uses Euclidean step length ||direction|| so diagonal and axis rays are comparable.
    """
    prof = ray_samples_abs(G, direction)
    step_len = float(math.sqrt(sum(s*s for s in direction)))
    return measure_mass_from_profile(prof, d=d, step_len=step_len, rmin=rmin, rmax=rmax, wmin=wmin, wmax=wmax, use_prefactor=True)

# Demo: synthetic 4D Yukawa tail (1D radial profile)
if RUN_MASS_DEMO:
    device_t = "cuda" if torch.cuda.is_available() else "cpu"
    dtype_t  = torch.float64
    torch.manual_seed(0)

    true_m = 0.55
    R = 80
    r = torch.arange(R, device=device_t, dtype=dtype_t)
    d = MASS_DIM

    # ideal Yukawa tail: exp(-m r)/r^{(d-1)/2} (use r+1 to avoid r=0)
    clean = torch.exp(-true_m * r) / (r + 1.0) ** (0.5 * (d - 1))
    noise = 1.0 + 0.01 * torch.randn_like(clean)
    profile = torch.abs(clean * noise)

    rep = measure_mass_from_profile(profile, d=d, step_len=1.0,
                                    rmin=MASS_RMIN, rmax=MASS_RMAX,
                                    wmin=MASS_WMIN, wmax=MASS_WMAX,
                                    use_prefactor=True)

    print("\n[DEMO] Mass module on synthetic 4D Yukawa tail")
    print(f"  true m={true_m:.6f}   measured m={rep['m']:.6f}  (MAD~{rep['mad']:.2e}, window {rep['window']})")

    # --- HOW TO APPLY TO YOUR INTERACTING CORRELATORS ---
    # Suppose you have a 4D lattice correlator tensor G[x0,x1,x2,x3] (already ensemble-averaged),
    # e.g. from gauge-projected kernel or Wilson loop correlator turned into a site correlator.
    # Then do, for example:
    #
    #   G = your_correlator_tensor  # torch.Tensor shape (L,L,L,L)
    #   for direction in [(1,0,0,0), (1,1,0,0), (1,1,1,0), (1,1,1,1)]:
    #       rep = measure_mass_yukawa_prefactor(G, direction, d=4, rmin=..., wmin=..., wmax=...)
    #       print(direction, rep)
    #
    # If the directions disagree, it’s usually anisotropy / finite-volume / operator overlap.

# ============================================================
# 2) SU(2) RG INTERTWINING TEST (INTRINSIC FIX)
#    Key fix: compute gradients w.r.t. algebra variables directly.
#    Expect: R_X ~ 1/n, R_Z ~ 4/n, ratio ~4.
# ============================================================
if RUN_RG_DEMO:
    try:
        import jax
        import jax.numpy as jnp
        from functools import partial

        # Optional: higher precision for the constant check
        jax.config.update("jax_enable_x64", True)

        def _qnormalize(q):
            return q / jnp.linalg.norm(q, axis=-1, keepdims=True)

        def _canonicalize(q):
            sgn = jnp.where(q[..., :1] < 0.0, -1.0, 1.0)
            return q * sgn

        def _qmul(a, b):
            aw, av = a[..., 0], a[..., 1:]
            bw, bv = b[..., 0], b[..., 1:]
            w = aw * bw - jnp.sum(av * bv, axis=-1)
            v = aw[..., None] * bv + bw[..., None] * av + jnp.cross(av, bv)
            return jnp.concatenate([w[..., None], v], axis=-1)

        def _qinv(q):  # for unit q
            return jnp.concatenate([q[..., :1], -q[..., 1:]], axis=-1)

        def su2_exp(X):
            """
            X in Lie algebra coords (R^3), returns quaternion exp(X/2).
            """
            X = jnp.asarray(X)
            theta = jnp.linalg.norm(X, axis=-1, keepdims=True)
            half = 0.5 * theta
            # sin(half)/theta ~ 1/2 as theta->0
            s_over_t = jnp.where(theta > 1e-12, jnp.sin(half) / theta, 0.5 - (theta**2)/48.0)
            w = jnp.cos(half)
            v = s_over_t * X
            q = jnp.concatenate([w, v], axis=-1)
            return _canonicalize(_qnormalize(q))

        def su2_log(q):
            """
            q quaternion on SU(2), returns Lie algebra coords (R^3).
            """
            q = _canonicalize(_qnormalize(q))
            w = q[..., 0]
            v = q[..., 1:]
            nv = jnp.linalg.norm(v, axis=-1)
            angle = 2.0 * jnp.arctan2(nv, w)  # in [0, pi]
            coef = jnp.where(nv > 1e-12, angle / nv, 2.0)
            return coef[..., None] * v

        def karcher_mean(U, iters: int):
            """
            Karcher mean on SU(2) using log/exp updates.
            U shape (n,4). iters is static.
            """
            U = _canonicalize(_qnormalize(U))
            q0 = U[0]

            def body(_, q):
                rel = _qmul(_qinv(q), U)          # (n,4)
                logs = su2_log(rel)               # (n,3)
                delta = jnp.mean(logs, axis=0)    # (3,)
                q_new = _qmul(q, su2_exp(delta))
                return _canonicalize(_qnormalize(q_new))

            return jax.lax.fori_loop(0, iters, body, q0)

        def pi_geodesic_from_X(X, iters: int):
            """
            X shape (n,3), interpreted as Lie algebra variables.
            """
            U = su2_exp(X)                # (n,4)
            qbar = karcher_mean(U, iters) # (4,)
            return su2_log(qbar)          # (3,)

        def _R_single(key, n: int, eps: float, iters: int):
            """
            Returns (R_intrinsic, R_qvec) on one random sample.
            - intrinsic: grad wrt X (Lie algebra coords)
            - qvec-like: grad wrt Z where U = exp((2Z)/2) = exp(Z) so Z ~ quaternion vector part near I
            """
            k1, k2, k3 = jax.random.split(key, 3)
            v = jax.random.normal(k1, (3,))
            X = eps * jax.random.normal(k2, (n, 3))
            Z = 0.5 * X  # SAME group element if we feed 2Z == X

            def fX(X_):
                Y = pi_geodesic_from_X(X_, iters)
                return jnp.dot(v, Y)

            def fZ(Z_):
                Y = pi_geodesic_from_X(2.0 * Z_, iters)  # same U as X when Z=X/2
                return jnp.dot(v, Y)

            gX = jax.grad(fX)(X)         # (n,3)
            gZ = jax.grad(fZ)(Z)         # (n,3)

            denom = jnp.sum(v * v) + 1e-30
            RX = jnp.sum(gX * gX) / denom
            RZ = jnp.sum(gZ * gZ) / denom
            return RX, RZ

        @partial(jax.jit, static_argnames=("n_samples", "n", "iters"))
        def batch_R_both(key, n_samples: int, n: int, eps: float, iters: int):
            keys = jax.random.split(key, n_samples)
            RX, RZ = jax.vmap(lambda kk: _R_single(kk, n=n, eps=eps, iters=iters))(keys)
            return RX, RZ

        key = jax.random.PRNGKey(0)
        RX, RZ = batch_R_both(key, n_samples=RG_NSAMPLES, n=RG_NLINKS, eps=RG_EPS, iters=RG_ITERS)

        def _stats(x):
            return dict(mean=float(jnp.mean(x)),
                        median=float(jnp.median(x)),
                        max=float(jnp.max(x)))

        stX = _stats(RX)
        stZ = _stats(RZ)
        ratio_Z_over_X = stZ["median"] / (stX["median"] + 1e-30)
        ratio_X_over_Z = stX["median"] / (stZ["median"] + 1e-30)

        print("\n[DEMO] SU(2) RG intertwining: intrinsic (algebra) vs qvec-like gradients")
        print(f"  intrinsic (X) R: mean={stX['mean']:.6f} median={stX['median']:.6f} max={stX['max']:.6f}")
        print(f"  qvec-like (Z) R: mean={stZ['mean']:.6f} median={stZ['median']:.6f} max={stZ['max']:.6f}")
        print(f"  Z/X median ratio ≈ {ratio_Z_over_X:.6f}   (expect ~4.0)")
        print(f"  X/Z median ratio ≈ {ratio_X_over_Z:.6f}   (expect ~0.25)")
        print(f"  expected intrinsic ~1/n = {1.0/RG_NLINKS:.6f}, expected qvec-like ~4/n = {4.0/RG_NLINKS:.6f}")

    except Exception as e:
        print("\n[SKIP] JAX not available or RG demo failed:", repr(e))

# ============================================================
# 3) SU(3) COMMUTATOR SEED: SURROGATE -> TRUE INVARIANT ||[A_mu,A_nu]||^2
# ============================================================
if RUN_SU3_DEMO:
    try:
        import jax
        import jax.numpy as jnp
        from functools import partial

        jax.config.update("jax_enable_x64", True)

        def build_su3_f_tensor():
            f = np.zeros((8,8,8), dtype=float)
            entries = [
                (1,2,3, 1.0),
                (1,4,7, 0.5),
                (1,5,6, 0.5),
                (2,4,6, 0.5),
                (2,5,7,-0.5),
                (3,4,5, 0.5),
                (3,6,7, 0.5),
                (4,5,8, math.sqrt(3)/2),
                (6,7,8, math.sqrt(3)/2),
            ]
            for (a,b,c,val) in entries:
                a-=1; b-=1; c-=1
                f[a,b,c] =  val
                f[b,c,a] =  val
                f[c,a,b] =  val
                f[a,c,b] = -val
                f[c,b,a] = -val
                f[b,a,c] = -val
            return jnp.array(f)

        f_su3 = build_su3_f_tensor()
        M_bc = jnp.sum(f_su3 * f_su3, axis=0)  # M_bc = Σ_a f_{abc}^2  (surrogate weight)

        def su3_comm_seed_surrogate(theta_mu, kappa):
            """
            Surrogate: kappa * Σ_{mu<nu} Σ_{b,c} M_bc * (A_mu^b)^2 * (A_nu^c)^2
            """
            A2 = theta_mu * theta_mu  # (4,8)
            acc = 0.0
            for mu in range(4):
                for nu in range(mu+1, 4):
                    acc = acc + jnp.einsum("b,c,bc->", A2[mu], A2[nu], M_bc)
            return kappa * acc

        def su3_comm_seed_true(theta_mu, kappa):
            """
            True invariant: kappa * Σ_{mu<nu} ||[A_mu, A_nu]||^2
            with [A_mu, A_nu]_a = Σ_{b,c} f_{a b c} A_mu^b A_nu^c
            """
            acc = 0.0
            for mu in range(4):
                for nu in range(mu+1, 4):
                    comm = jnp.einsum("abc,b,c->a", f_su3, theta_mu[mu], theta_mu[nu])  # (8,)
                    acc = acc + jnp.sum(comm * comm)
            return kappa * acc

        def hess_eigs(seed_fn, x_flat):
            H = jax.hessian(lambda z: seed_fn(z.reshape(4,8)))(x_flat)
            # force sym (numerical)
            H = 0.5 * (H + H.T)
            evals = jnp.linalg.eigvalsh(H)
            return evals

        def min_eig(seed_fn, x_flat):
            return hess_eigs(seed_fn, x_flat)[0]

        def safe_radius_proxy(seed_fn, n_dirs=10, eps_grid=(0.0, 0.5, 1.0), tol=1e-10, seed=0):
            key = jax.random.PRNGKey(seed)
            key, kdir = jax.random.split(key)
            dirs = jax.random.normal(kdir, (n_dirs, 32))
            dirs = dirs / (jnp.linalg.norm(dirs, axis=-1, keepdims=True) + 1e-30)

            eps_grid = jnp.array(eps_grid)
            safes = []
            for i in range(n_dirs):
                u = dirs[i]
                ok = []
                for eps in eps_grid:
                    x = eps * u
                    ok.append(float(min_eig(seed_fn, x) > tol))
                # largest eps that is "ok"
                last_ok = 0.0
                for eps, flag in zip(list(map(float, eps_grid)), ok):
                    if flag >= 0.5:
                        last_ok = eps
                safes.append(last_ok)
            safes = np.array(safes, dtype=float)
            return float(np.median(safes)), safes

        # Compare at SU3_EPS0
        key = jax.random.PRNGKey(123)
        theta0 = SU3_EPS0 * jax.random.normal(key, (4,8))
        x0 = theta0.reshape(-1)

        evals_true = hess_eigs(lambda th: su3_comm_seed_true(th, SU3_KAPPA), x0)
        evals_sur  = hess_eigs(lambda th: su3_comm_seed_surrogate(th, SU3_KAPPA), x0)

        print("\n[DEMO] SU(3) commutator seed: surrogate vs true (Hessian + safe-radius proxy)")
        print(f"  At eps={SU3_EPS0:.3f}: min eig true={float(evals_true[0]):.6f}, surrogate={float(evals_sur[0]):.6f}")
        print(f"    true eig[0:6]={np.array(evals_true[:6])}")
        print(f"    sur  eig[0:6]={np.array(evals_sur[:6])}")

        med_true, arr_true = safe_radius_proxy(lambda th: su3_comm_seed_true(th, SU3_KAPPA),
                                               n_dirs=SU3_NDIRS, eps_grid=SU3_EPS_GRID, tol=1e-10, seed=1)
        med_sur,  arr_sur  = safe_radius_proxy(lambda th: su3_comm_seed_surrogate(th, SU3_KAPPA),
                                               n_dirs=SU3_NDIRS, eps_grid=SU3_EPS_GRID, tol=1e-10, seed=2)

        print(f"  grid-safe eps ({SU3_NDIRS} dirs): median true={med_true:.3f}, surrogate={med_sur:.3f}")
        print("  (Single-site proxy only; full lattice safe region will be smaller.)")

    except Exception as e:
        print("\n[SKIP] JAX not available or SU3 demo failed:", repr(e))

# ============================================================
# 4) CasRG(model2): swap full-reorth Lanczos -> LOBPCG (fast) / eigsh / shift-invert
#    (This is your one-shot driver rewritten cleanly with solver swap.)
# ============================================================
if RUN_CASRG_MODEL2:
    import time
    import scipy.sparse as sp
    import scipy.sparse.linalg as spla

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype  = torch.float64
    print("\n[CasRG] device:", device)

    # ---------------------------- Config ----------------------------
    L = int(CASRG_L)
    alpha_list = list(CASRG_ALPHA_LIST)
    beta_list  = list(CASRG_BETA_LIST)
    eps_list   = list(CASRG_EPS_BITS)

    g = 1.0
    k_eigs = int(CASRG_K_EIGS)

    # ---------------------------- SU(3) reps p+q<=2 ----------------------------
    REPS = [(0,0),(1,0),(0,1),(2,0),(0,2),(1,1)]
    REP_ID = {rq:i for i,rq in enumerate(REPS)}
    def dual(r): return (r[1], r[0])
    def C2(p,q): return (p*p + q*q + p*q + 3*p + 3*q)/3.0
    C2_table = {REP_ID[r]: C2(*r) for r in REPS}

    def trunc(components):
        return tuple(sorted({c for c in components if c in REP_ID}))

    TP = {}
    def add_tp(a,b,comps): TP[(a,b)] = trunc(comps)
    add_tp((1,0),(1,0), [(2,0),(0,1)])
    add_tp((1,0),(0,1), [(0,0),(1,1)])
    add_tp((0,1),(0,1), [(0,2),(1,0)])
    add_tp((2,0),(1,0), [(1,1)])
    add_tp((2,0),(0,1), [(1,0)])
    add_tp((0,2),(0,1), [(1,1)])
    add_tp((0,2),(1,0), [(0,1)])
    add_tp((1,1),(1,0), [(1,0),(2,0)])
    add_tp((1,1),(0,1), [(0,1),(0,2)])
    add_tp((1,1),(1,1), [(0,0),(1,1)])
    add_tp((2,0),(0,2), [(0,0),(1,1)])
    add_tp((2,0),(2,0), [])
    add_tp((0,2),(0,2), [])
    add_tp((2,0),(1,1), [(2,0)])
    add_tp((0,2),(1,1), [(0,2)])

    for (a,b), comps in list(TP.items()):
        if (b,a) not in TP:
            TP[(b,a)] = comps

    def tp(a,b):
        if (a,b) in TP:
            return TP[(a,b)]
        da, db = dual(a), dual(b)
        if (da,db) in TP:
            return tuple(sorted(dual(r) for r in TP[(da,db)]))
        return tuple()

    def truncated_tensor_closure(rep_list):
        if not rep_list:
            return {(0,0)}
        S = {rep_list[0]}
        for r in rep_list[1:]:
            S2 = set()
            for s in S:
                for c in tp(s,r):
                    S2.add(c)
            S = S2
            if not S:
                return set()
        return S

    # ---------------------------- Lattice (2D torus) ----------------------------
    def link_id(x,y,mu): return (x % L, y % L, mu)
    V = [(x % L, y % L) for x in range(L) for y in range(L)]
    E = [link_id(x,y,mu) for x in range(L) for y in range(L) for mu in (0,1)]
    E_index = {e:i for i,e in enumerate(E)}
    P = [(x % L, y % L) for x in range(L) for y in range(L)]
    P_index = {p:i for i,p in enumerate(P)}
    n_plaq = len(P)

    def outgoing_edges(v):
        x,y = v
        return [link_id(x,y,0), link_id(x,y,1)]

    def incoming_edges(v):
        x,y = v
        return [link_id(x-1,y,0), link_id(x,y-1,1)]

    def vertex_gauss_ok(rep_on_links):
        for v in V:
            reps = []
            for e in outgoing_edges(v):
                reps.append(REPS[rep_on_links[E_index[e]]])
            for e in incoming_edges(v):
                reps.append(dual(REPS[rep_on_links[E_index[e]]]))
            if (0,0) not in truncated_tensor_closure(reps):
                return False
        return True

    FUND, AFUND = (1,0), (0,1)
    def fuse_channels(r,f): return tp(r,f)

    def apply_plaquette_branches(rep_on_links, p_xy):
        x,y = p_xy
        e1 = link_id(x,y,0)
        e2 = link_id(x+1,y,1)
        e3 = link_id(x,y+1,0)  # backward => AFUND
        e4 = link_id(x,y,1)    # backward => AFUND

        reps = [REPS[rep_on_links[E_index[e]]] for e in (e1,e2,e3,e4)]
        chans = [
            fuse_channels(reps[0], FUND),
            fuse_channels(reps[1], FUND),
            fuse_channels(reps[2], AFUND),
            fuse_channels(reps[3], AFUND),
        ]
        if any(len(c)==0 for c in chans):
            return []
        out = []
        for r1,r2,r3,r4 in itertools.product(*chans):
            rep2 = list(rep_on_links)
            rep2[E_index[e1]] = REP_ID[r1]
            rep2[E_index[e2]] = REP_ID[r2]
            rep2[E_index[e3]] = REP_ID[r3]
            rep2[E_index[e4]] = REP_ID[r4]
            out.append(tuple(rep2))
        return out

    # ---------------------------- Build bases ----------------------------
    all_rep_ids = list(range(len(REPS)))
    print("[CasRG] Building link basis...")
    t0 = time.time()
    link_basis = []
    for assign in itertools.product(all_rep_ids, repeat=len(E)):
        if vertex_gauss_ok(assign):
            link_basis.append(tuple(assign))
    print("[CasRG] link basis size:", len(link_basis), f"(t={time.time()-t0:.2f}s)")

    bit_basis = list(itertools.product([0,1], repeat=n_plaq))
    print("[CasRG] Building extended basis...")
    t0 = time.time()
    basis = [(st,bits) for st in link_basis for bits in bit_basis]
    nb = len(basis)
    basis_index = {st:i for i,st in enumerate(basis)}
    print("[CasRG] extended basis size:", nb, f"(t={time.time()-t0:.2f}s)")

    # Precompute diagonal pieces independent of beta/eps
    bitcount = np.zeros(nb, dtype=np.float64)
    Ediag    = np.zeros(nb, dtype=np.float64)
    for i,(links,bits) in enumerate(basis):
        bitcount[i] = float(sum(bits))
        Ediag[i] = 0.5*(g**2) * sum(C2_table[rid] for rid in links)

    def symmetrize_sparse(H):
        H = H.coalesce()
        idx = H.indices()
        val = H.values()
        HT = torch.sparse_coo_tensor(
            torch.stack([idx[1], idx[0]], dim=0),
            val,
            H.shape,
            device=H.device,
            dtype=H.dtype
        ).coalesce()
        H2 = (H + HT).coalesce()
        return torch.sparse_coo_tensor(H2.indices(), 0.5 * H2.values(), H2.shape, device=H.device, dtype=H.dtype).coalesce()

    # --- EIGENSOLVERS ---
    @torch.no_grad()
    def lowest_eigs_lobpcg(Hs, k=4, tol=1e-8, niter=250, seed=0):
        torch.manual_seed(seed)
        n = Hs.shape[0]
        X = torch.randn((n, k), device=Hs.device, dtype=Hs.dtype)
        Q, _ = torch.linalg.qr(X)
        evals, _ = torch.lobpcg(Hs, Q, largest=False, tol=tol, maxiter=niter)
        out = evals.detach().cpu().numpy()
        out.sort()
        return out

    def _torch_sparse_to_scipy_csr(Hs):
        Hs = Hs.coalesce().cpu()
        idx = Hs.indices().numpy()
        val = Hs.values().numpy()
        n = Hs.shape[0]
        A = sp.coo_matrix((val, (idx[0], idx[1])), shape=(n, n)).tocsr()
        return A

    def lowest_eigs_eigsh_cpu(Hs, k=4, tol=1e-10, which="SA", maxiter=None):
        A = _torch_sparse_to_scipy_csr(Hs)
        evals = spla.eigsh(A, k=k, which=which, tol=tol, maxiter=maxiter, return_eigenvectors=False)
        evals = np.array(evals, dtype=float)
        evals.sort()
        return evals

    def lowest_eigs_shiftinvert_cpu(Hs, k=4, sigma=0.0, tol=1e-12, maxiter=None):
        A = _torch_sparse_to_scipy_csr(Hs)
        # eigenvalues nearest sigma
        evals = spla.eigsh(A, k=k, sigma=sigma, which="LM", tol=tol, maxiter=maxiter, return_eigenvectors=False)
        evals = np.array(evals, dtype=float)
        evals.sort()
        return evals

    def lowest_eigs(Hs, solver="lobpcg", k=4, tol=1e-8, niter=250, seed=0):
        if solver == "lobpcg":
            try:
                return lowest_eigs_lobpcg(Hs, k=k, tol=tol, niter=niter, seed=seed)
            except Exception as e:
                print("[CasRG] LOBPCG failed, falling back to eigsh:", repr(e))
                return lowest_eigs_eigsh_cpu(Hs, k=k, tol=tol)
        elif solver == "eigsh":
            return lowest_eigs_eigsh_cpu(Hs, k=k, tol=tol)
        elif solver == "shift-invert":
            return lowest_eigs_shiftinvert_cpu(Hs, k=k, sigma=0.0, tol=tol)
        else:
            raise ValueError(f"Unknown solver={solver}")

    def gap_from_list(evals):
        e0 = float(evals[0])
        for ev in evals[1:]:
            if float(ev) > e0 + 1e-10:
                return e0, float(ev - e0)
        return e0, float("nan")

    # Pre-build diag index
    diag_idx  = torch.arange(nb, device=device, dtype=torch.int64)
    diag_idx2 = torch.stack([diag_idx, diag_idx], dim=0)

    # Off-template builder per alpha (unit beta, no diag)
    def build_off_template(alpha):
        acc = defaultdict(float)
        for i,(links,bits) in enumerate(basis):
            for p_xy in P:
                pidx = P_index[p_xy]
                branches = apply_plaquette_branches(links, p_xy)
                if not branches:
                    continue
                bits_toggled = list(bits)
                bits_toggled[pidx] = 1 - bits_toggled[pidx]
                bits_toggled = tuple(bits_toggled)

                js = []
                ws = []
                for links2 in branches:
                    j = basis_index.get((links2, bits_toggled), None)
                    if j is None:
                        continue
                    s = 0.0
                    for rid in links2:
                        s += C2_table[rid]
                    w = math.exp(-alpha * s)
                    js.append(j); ws.append(w)

                if not ws:
                    continue
                Z = sum(ws)
                invZ = 1.0 / Z
                for j,w in zip(js,ws):
                    acc[(i,j)] += -(w * invZ)  # unit beta

        keys = list(acc.keys())
        rows = np.fromiter((k[0] for k in keys), dtype=np.int64, count=len(keys))
        cols = np.fromiter((k[1] for k in keys), dtype=np.int64, count=len(keys))
        vals = np.fromiter((acc[k] for k in keys), dtype=np.float64, count=len(keys))
        off_idx = torch.stack([torch.from_numpy(rows), torch.from_numpy(cols)], dim=0).to(device=device)
        off_val_unit = torch.from_numpy(vals).to(device=device, dtype=dtype)
        return off_idx, off_val_unit

    results = []
    for alpha in alpha_list:
        print(f"\n[CasRG] === alpha={alpha} building off-template ===")
        t0 = time.time()
        off_idx, off_val_unit = build_off_template(alpha)
        print("[CasRG] off nnz =", off_val_unit.numel(), f"(t={time.time()-t0:.2f}s)")

        for beta in beta_list:
            print(f"\n[CasRG] --- alpha={alpha} beta={beta} ---")
            off_val = beta * off_val_unit
            diag_base = Ediag + beta * n_plaq

            for eps_bit in eps_list:
                diag = diag_base + eps_bit * bitcount
                diag_val = torch.from_numpy(diag).to(device=device, dtype=dtype)

                idx_all = torch.cat([off_idx, diag_idx2], dim=1)
                val_all = torch.cat([off_val, diag_val], dim=0)

                Hs = torch.sparse_coo_tensor(idx_all, val_all, (nb, nb), device=device, dtype=dtype).coalesce()
                Hs = symmetrize_sparse(Hs)

                t1 = time.time()
                evals = lowest_eigs(Hs, solver=CASRG_SOLVER, k=k_eigs, tol=CASRG_TOL, niter=CASRG_NITER, seed=CASRG_SEED)
                e0, gap = gap_from_list(evals)
                results.append((alpha, beta, eps_bit, e0, gap))
                print(f"[CasRG] eps={eps_bit:.1e} e0={e0:.6f} gap={gap:.6f}   (solver={CASRG_SOLVER}, t={time.time()-t1:.2f}s)")

    out_path = "/mnt/data/casrg_model2_results_lobpcg.csv"
    with open(out_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["alpha","beta","eps_bit","e0","gap"])
        w.writerows(results)

    print("\n[CasRG] Wrote:", out_path)
    print("[CasRG] Done.")

# ============================================================
# DONE
# ============================================================
print("\n[DONE]")
print(" - Mass module: measure_mass_from_profile(...) and measure_mass_yukawa_prefactor(G, direction, ...)")
print(" - SU(2) RG: batch_R_both(...) prints intrinsic (1/n) and qvec-like (4/n) + ratio ~4")
print(" - SU(3): su3_comm_seed_true(...) vs surrogate, Hessian spectra + safe-radius proxy")
print(" - CasRG: LOBPCG swap + eps_bit scan (enable RUN_CASRG_MODEL2=True)")



[DEMO] Mass module on synthetic 4D Yukawa tail
  true m=0.550000   measured m=0.554332  (MAD~7.70e-04, window (54.5, 61.5))

[DEMO] SU(2) RG intertwining: intrinsic (algebra) vs qvec-like gradients
  intrinsic (X) R: mean=nan median=nan max=nan
  qvec-like (Z) R: mean=nan median=nan max=nan
  Z/X median ratio ≈ nan   (expect ~4.0)
  X/Z median ratio ≈ nan   (expect ~0.25)
  expected intrinsic ~1/n = 0.062500, expected qvec-like ~4/n = 0.250000

[DEMO] SU(3) commutator seed: surrogate vs true (Hessian + safe-radius proxy)
  At eps=0.350: min eig true=-0.344992, surrogate=-0.234127
    true eig[0:6]=[-0.3449915  -0.32829774 -0.18400378 -0.05735051  0.00271863  0.03645132]
    sur  eig[0:6]=[-0.23412694  0.0033526   0.08745378  0.13860612  0.1895012   0.23625764]
  grid-safe eps (12 dirs): median true=0.000, surrogate=0.000
  (Single-site proxy only; full lattice safe region will be smaller.)

[DONE]
 - Mass module: measure_mass_from_profile(...) and measure_mass_yukawa_prefactor(G, dire

In [7]:
# ================================================================
# CLEAN BLOCK v2 (Mass module + SU(2) RG intertwining + SU(3) comm seed + CasRG solver patch)
#   Goals:
#     (1) Mass extraction on torus with prefactor-corrected Yukawa tail
#     (2) SU(2) RG intertwining constant in *intrinsic algebra norms* vs qvec-like coords (factor-of-4 trap)
#     (3) SU(3) commutator seed: surrogate ("square-of-squares") -> true ||[Aμ,Aν]||^2, compare Hessians + safe proxy
#     (4) CasRG model2: drop-in eigen solver swap (LOBPCG / eigsh / shift-invert) + scan harness
#
# If you previously got NaNs:
#   - SU(2) RG: NaNs almost always mean your sampled quaternions wandered near the cut-locus (w≈-1, |v|≈0)
#     OR your qvec parametrization violated |z|<1 so sqrt(1-|z|^2) went invalid.
#     This block clamps z safely and keeps eps small by default.
#   - SU(3) comm seed: negative Hessian eigenvalues can be real (seed is a sum of squares but not convex),
#     BUT if you see wild sign flips, confirm you used +kappa*(...) not -kappa*(...).
# ================================================================

import math
import numpy as np

# -------------------------
# Optional deps (CasRG patch uses SciPy; SU(2)/SU(3) tests use JAX)
# -------------------------
try:
    import jax
    import jax.numpy as jnp
    from functools import partial
    JAX_OK = True
except Exception as e:
    JAX_OK = False
    print("[WARN] JAX not available in this environment. SU(2)/SU(3) demos will be skipped.")
    print("       Error:", repr(e))

try:
    import scipy
    import scipy.sparse as sp
    import scipy.sparse.linalg as spla
    SCIPY_OK = True
except Exception as e:
    SCIPY_OK = False
    print("[WARN] SciPy not available in this environment. CasRG solver demo/patch will be skipped.")
    print("       Error:", repr(e))


# ================================================================
# 1) MASS MEASUREMENT MODULE (prefactor-corrected Yukawa tail)
# ================================================================

def _mad(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    med = np.median(x)
    return float(np.median(np.abs(x - med)))

def _direction_norm2(direction):
    direction = tuple(int(v) for v in direction)
    return sum(v*v for v in direction)

def profile_from_lattice_along_direction(G_lat: np.ndarray, direction=(1,0,0,0), origin=None, n_max=None):
    """
    Very lightweight extractor: samples G at points n*direction (mod L) from a single origin.
    If your correlator is already translationally averaged (typical), this is fine.
    Otherwise you may want to average over many origins yourself.
    """
    G_lat = np.asarray(G_lat)
    d = G_lat.ndim
    L = G_lat.shape[0]
    assert all(G_lat.shape[i] == L for i in range(d)), "Expected an L^d torus array."

    direction = np.array([int(v) for v in direction], dtype=int)
    if origin is None:
        origin = np.zeros(d, dtype=int)
    else:
        origin = np.array(origin, dtype=int)

    if n_max is None:
        n_max = L // 2

    prof = np.zeros(n_max + 1, dtype=float)
    for n in range(n_max + 1):
        x = (origin + n * direction) % L
        prof[n] = float(G_lat[tuple(x.tolist())])
    return prof

def measure_mass_yukawa_prefactor(
    G_1d,
    L: int,
    direction=(1,0,0,0),
    d: int = 4,
    rmin: int = 10,
    W: int = 8,
    tail_frac: float = 0.40,
    tiny: float = 1e-300,
):
    """
    Mass from a 1D profile on a dD torus, along a lattice direction.

    Model: G(r) ~ C * r^{-(d-1)/2} * exp(-m r)   (large r)
    => y(r) := log|G(r)| + ((d-1)/2)*log r  has slope ~ -m.

    Returns:
      m_hat, mad_tail, window=(n_start+0.5, n_start+W+0.5), plus diagnostics dict.
    """
    G = np.asarray(G_1d, dtype=float)
    n_max = min(len(G) - 1, L // 2)

    direction = tuple(int(v) for v in direction)
    k = _direction_norm2(direction)
    if k <= 0:
        raise ValueError("direction must have at least one nonzero component.")
    step = math.sqrt(k)

    # radii in Euclidean units along this direction (valid up to L/2)
    ns = np.arange(0, n_max + 1)
    r = step * ns
    r = np.maximum(r, 1e-12)

    A = np.abs(G[: n_max + 1])
    y = np.log(np.maximum(A, tiny)) + 0.5 * (d - 1) * np.log(r)

    # Local windowed slopes: average of finite differences in y divided by dr
    # m_hat(R) corresponds to window ns in [R, R+W]
    R_lo = max(int(rmin), 1)
    R_hi = n_max - W - 1
    if R_hi <= R_lo:
        raise ValueError(f"Not enough points: n_max={n_max}, rmin={rmin}, W={W}")

    Rs = np.arange(R_lo, R_hi + 1)
    m_hats = []
    for R0 in Rs:
        yy = y[R0 : R0 + W + 1]
        rr = r[R0 : R0 + W + 1]
        dy = np.diff(yy)
        dr = np.diff(rr)
        mloc = -float(np.mean(dy / dr))
        m_hats.append(mloc)
    m_hats = np.array(m_hats, dtype=float)

    # Tail-based robust plateau pick:
    #   define tail region = last tail_frac of m_hats
    t0 = int((1.0 - float(tail_frac)) * len(m_hats))
    t0 = max(0, min(t0, len(m_hats) - 1))
    tail = m_hats[t0:]
    plateau = float(np.median(tail))
    mad_tail = _mad(tail)

    # Choose the window whose m_hat is closest to the tail-median plateau
    i_best_in_tail = int(np.argmin(np.abs(tail - plateau)))
    i_best = t0 + i_best_in_tail
    m_best = float(m_hats[i_best])
    R_best = int(Rs[i_best])

    window = (R_best + 0.5, R_best + W + 0.5)

    diag = {
        "Rs": Rs,
        "m_hats": m_hats,
        "plateau_tail_median": plateau,
        "tail_mad": mad_tail,
        "R_best": R_best,
        "step_Euclid": step,
    }
    return m_best, mad_tail, window, diag


def demo_mass_module_synthetic_4d_yukawa():
    # Synthetic 4D Yukawa tail on a torus direction (axis)
    L = 128
    d = 4
    true_m = 0.55
    direction = (1,0,0,0)
    step = 1.0

    n_max = L // 2
    ns = np.arange(0, n_max + 1)
    r = np.maximum(step * ns, 1e-12)

    # synthetic tail + mild noise
    rng = np.random.default_rng(0)
    C = 2.0
    G = C * np.exp(-true_m * r) / (r ** ((d - 1) / 2))
    G[0] = C  # avoid singularity at r=0 for the synthetic construction
    G = G * (1.0 + 0.002 * rng.normal(size=G.shape))

    m_hat, mad, window, _ = measure_mass_yukawa_prefactor(
        G, L=L, direction=direction, d=d, rmin=14, W=7, tail_frac=0.35
    )
    print("[DEMO] Mass module on synthetic 4D Yukawa tail")
    print(f"  true m={true_m:.6f}   measured m={m_hat:.6f}  (MAD~{mad:.2e}, window {window})")
    print()


# ================================================================
# 2) SU(2) RG INTERTWINING TEST (intrinsic algebra norm vs qvec-like coords)
# ================================================================
if JAX_OK:
    try:
        jax.config.update("jax_enable_x64", True)
    except Exception:
        pass

    def qconj(q):
        return jnp.concatenate([q[..., :1], -q[..., 1:]], axis=-1)

    def qnormalize(q, eps=1e-15):
        n = jnp.linalg.norm(q, axis=-1, keepdims=True)
        return q / jnp.maximum(n, eps)

    def qmul(a, b):
        aw, av = a[..., :1], a[..., 1:]
        bw, bv = b[..., :1], b[..., 1:]
        w = aw * bw - jnp.sum(av * bv, axis=-1, keepdims=True)
        v = aw * bv + bw * av + jnp.cross(av, bv)
        return jnp.concatenate([w, v], axis=-1)

    def su2_exp(X, eps=1e-12):
        # X: (...,3) algebra coords, quaternion = (cos(|X|/2), sin(|X|/2)/|X| * X)
        th = jnp.linalg.norm(X, axis=-1, keepdims=True)
        half = 0.5 * th
        w = jnp.cos(half)

        # sin(half)/th with Taylor near 0
        # sin(th/2)/th ~ 1/2 - th^2/48 + ...
        t = th
        coeff_taylor = 0.5 - (t * t) / 48.0
        coeff = jnp.where(t < 1e-6, coeff_taylor, jnp.sin(half) / jnp.maximum(t, eps))
        v = coeff * X
        return qnormalize(jnp.concatenate([w, v], axis=-1))

    def su2_log(q, eps=1e-12):
        # q: (...,4) unit quaternion -> X: (...,3)
        q = qnormalize(q)
        w = jnp.clip(q[..., 0], -1.0, 1.0)
        v = q[..., 1:]
        nv = jnp.linalg.norm(v, axis=-1)

        theta = 2.0 * jnp.arctan2(nv, w)  # in [0, 2pi]
        # near identity: X ~ 2 v
        coeff = jnp.where(nv < 1e-6, 2.0, theta / jnp.maximum(nv, eps))
        return coeff[..., None] * v

    def karcher_mean_su2(qs, iters=6):
        # qs: (n,4)
        qbar = qs[0]
        for _ in range(iters):
            dq = qmul(qconj(qbar), qs)   # qbar^{-1} * qi
            Xi = su2_log(dq)             # (n,3)
            delta = jnp.mean(Xi, axis=0) # (3,)
            qbar = qmul(qbar, su2_exp(delta))
            qbar = qnormalize(qbar)
        return qbar

    def pi_geodesic(qs, iters=6):
        # SU(2)^n -> su(2) (R^3): log(KarcherMean)
        return su2_log(karcher_mean_su2(qs, iters=iters))

    def _clamp_qvec(Z, zmax=0.999):
        # Ensure ||Z|| < 1 so w = sqrt(1-||Z||^2) is real.
        n = jnp.linalg.norm(Z, axis=-1, keepdims=True)
        scale = jnp.minimum(1.0, zmax / jnp.maximum(n, 1e-12))
        return Z * scale

    def qs_from_qvec(Z):
        Z = _clamp_qvec(Z)
        z2 = jnp.sum(Z * Z, axis=-1, keepdims=True)
        w = jnp.sqrt(jnp.maximum(1.0 - z2, 1e-15))
        return qnormalize(jnp.concatenate([w, Z], axis=-1))

    def R_intrinsic_from_algebra(Xs, v, iters=6):
        # Xs: (n,3) algebra coords
        def f_of_X(Xflat):
            X = Xflat.reshape(Xs.shape)
            qs = jax.vmap(su2_exp)(X)
            Y = pi_geodesic(qs, iters=iters)
            return jnp.dot(v, Y)
        g = jax.grad(f_of_X)(Xs.reshape(-1)).reshape(Xs.shape)
        num = jnp.sum(g * g)
        den = jnp.sum(v * v)
        return num / jnp.maximum(den, 1e-18)

    def R_qvec_like_from_Z(Zs, v, iters=6):
        # Zs: (n,3) treated as "quaternion vector coords"
        def f_of_Z(Zflat):
            Z = Zflat.reshape(Zs.shape)
            qs = qs_from_qvec(Z)
            Y = pi_geodesic(qs, iters=iters)
            return jnp.dot(v, Y)
        g = jax.grad(f_of_Z)(Zs.reshape(-1)).reshape(Zs.shape)
        num = jnp.sum(g * g)
        den = jnp.sum(v * v)
        return num / jnp.maximum(den, 1e-18)

    @partial(jax.jit, static_argnames=("n","nsamp","iters"))
    def batch_R_both(key, n=16, nsamp=256, iters=6, eps=0.08):
        # Near-identity samples in algebra coords:
        # Xs ~ eps * N(0,1), and Zs approximates qvec coords near identity.
        keyX, keyV = jax.random.split(key, 2)
        X = eps * jax.random.normal(keyX, shape=(nsamp, n, 3))
        V = jax.random.normal(keyV, shape=(nsamp, 3))
        # normalize v to avoid tiny denom edge cases
        V = V / jnp.maximum(jnp.linalg.norm(V, axis=-1, keepdims=True), 1e-12)

        # For qvec-like test, use Z = imag(q) = sin(|X|/2)/|X| * X
        # Near 0, Z ~ X/2, which is exactly where the factor-of-4 comes from.
        th = jnp.linalg.norm(X, axis=-1, keepdims=True)
        half = 0.5 * th
        coeff = jnp.where(th < 1e-6, 0.5 - (th*th)/48.0, jnp.sin(half) / jnp.maximum(th, 1e-12))
        Z = coeff * X

        Rin = jax.vmap(lambda xs, v: R_intrinsic_from_algebra(xs, v, iters=iters))(X, V)
        Rz  = jax.vmap(lambda zs, v: R_qvec_like_from_Z(zs, v, iters=iters))(Z, V)

        return Rin, Rz

    def demo_su2_rg_intertwining():
        key = jax.random.PRNGKey(0)
        n = 16
        Rin, Rz = batch_R_both(key, n=n, nsamp=128, iters=6, eps=0.08)

        # Robust stats + NaN check
        Rin_np = np.array(Rin)
        Rz_np  = np.array(Rz)
        bad = np.isnan(Rin_np).sum() + np.isnan(Rz_np).sum()
        if bad > 0:
            print("[WARN] NaNs detected in SU(2) RG demo. Reduce eps (try 0.05) and/or increase iters (try 8).")

        def stat(x):
            x = x[np.isfinite(x)]
            if len(x) == 0:
                return float("nan"), float("nan"), float("nan")
            return float(np.mean(x)), float(np.median(x)), float(np.max(x))

        m_in, med_in, mx_in = stat(Rin_np)
        m_z,  med_z,  mx_z  = stat(Rz_np)

        print("[DEMO] SU(2) RG intertwining: intrinsic (algebra) vs qvec-like gradients")
        print(f"  intrinsic (X) R: mean={m_in:.6f} median={med_in:.6f} max={mx_in:.6f}")
        print(f"  qvec-like (Z) R: mean={m_z:.6f} median={med_z:.6f} max={mx_z:.6f}")
        print(f"  Z/X median ratio ≈ {med_z/med_in:.6f}   (expect ~4.0)")
        print(f"  X/Z median ratio ≈ {med_in/med_z:.6f}   (expect ~0.25)")
        print(f"  expected intrinsic ~1/n = {1.0/n:.6f}, expected qvec-like ~4/n = {4.0/n:.6f}")
        print()


# ================================================================
# 3) SU(3) COMMUTATOR SEED: SURROGATE -> TRUE INVARIANT, HESSIAN + SAFE PROXY
# ================================================================
if JAX_OK:
    def build_su3_f_tensor():
        # Standard SU(3) antisymmetric structure constants f_{abc} (Gell-Mann basis).
        # Using the compact entry list you already had in your codebase.
        f = np.zeros((8, 8, 8), dtype=float)
        entries = [
            (1,2,3, 1.0),
            (1,4,7, 0.5),
            (1,5,6, 0.5),
            (2,4,6, 0.5),
            (2,5,7,-0.5),
            (3,4,5, 0.5),
            (3,6,7, 0.5),
            (4,5,8, math.sqrt(3)/2),
            (6,7,8, math.sqrt(3)/2),
        ]
        for (a,b,c,val) in entries:
            a -= 1; b -= 1; c -= 1
            f[a,b,c] =  val
            f[b,c,a] =  val
            f[c,a,b] =  val
            f[a,c,b] = -val
            f[c,b,a] = -val
            f[b,a,c] = -val
        return jnp.array(f, dtype=jnp.float64)

    f_su3 = build_su3_f_tensor()

    def su3_comm_seed_true(theta_mu, kappa):
        """
        theta_mu: (...,4,8) real adjoint coords for A_mu.
        seed = kappa * sum_{mu<nu} ||[A_mu, A_nu]||^2
        with [A_mu,A_nu]_a = sum_{b,c} f_{a b c} A_mu^b A_nu^c
        """
        A = theta_mu
        total = 0.0
        for mu in range(4):
            for nu in range(mu+1, 4):
                comm = jnp.einsum("abc,...b,...c->...a", f_su3, A[..., mu, :], A[..., nu, :])
                total = total + jnp.sum(comm * comm, axis=-1)
        return kappa * total

    def su3_comm_seed_surrogate(theta_mu, kappa):
        """
        Surrogate ("square-of-squares"):
          replace (sum_{b,c} f_{a b c} A_mu^b A_nu^c)^2
          by sum_{b,c} f_{a b c}^2 (A_mu^b)^2 (A_nu^c)^2
        then sum over a and mu<nu.
        """
        A = theta_mu
        f2 = f_su3 * f_su3
        total = 0.0
        for mu in range(4):
            for nu in range(mu+1, 4):
                tmp = jnp.einsum("abc,...b,...c->...a", f2, A[..., mu, :]**2, A[..., nu, :]**2)
                total = total + jnp.sum(tmp, axis=-1)
        return kappa * total

    def su3_total_energy(theta_flat, kappa, kind="true"):
        # Baseline quadratic so Hessian near ~I when fields are small (matches your earlier style).
        theta_mu = theta_flat.reshape((4,8))
        quad = 0.5 * jnp.sum(theta_mu * theta_mu)
        if kind == "true":
            return quad + su3_comm_seed_true(theta_mu, kappa)
        elif kind == "surrogate":
            return quad + su3_comm_seed_surrogate(theta_mu, kappa)
        else:
            raise ValueError("kind must be 'true' or 'surrogate'")

    def hessian_eigs_at(theta_mu, kappa, kind="true"):
        th = theta_mu.reshape(-1)
        H = jax.hessian(lambda x: su3_total_energy(x, kappa, kind=kind))(th)
        # Symmetrize numerically
        H = 0.5 * (H + H.T)
        evals = jnp.linalg.eigvalsh(H)
        return evals

    def safe_radius_proxy(theta_dir, kappa, kind="true", eps_grid=None):
        """
        Single-site proxy:
          Along a fixed direction in parameter space, find largest eps in grid
          such that min eig(H(theta=eps*dir)) >= 0.
        """
        if eps_grid is None:
            eps_grid = np.linspace(0.0, 1.0, 41)  # 0..1 step 0.025
        d = theta_dir / np.maximum(np.linalg.norm(theta_dir), 1e-12)
        best = 0.0
        for eps in eps_grid:
            theta = eps * d
            evals = np.array(hessian_eigs_at(jnp.array(theta.reshape(4,8)), kappa, kind=kind))
            if np.min(evals) >= -1e-10:
                best = float(eps)
        return best

    def demo_su3_comm_seed():
        key = jax.random.PRNGKey(0)
        kappa = 0.7
        eps = 0.35
        theta0 = jax.random.normal(key, shape=(4,8), dtype=jnp.float64)
        theta = eps * theta0

        evals_true = np.array(hessian_eigs_at(theta, kappa, kind="true"))
        evals_sur  = np.array(hessian_eigs_at(theta, kappa, kind="surrogate"))

        # Safe radius proxy: a handful of random directions
        rng = np.random.default_rng(0)
        dirs = rng.normal(size=(12, 32))
        sr_true = [safe_radius_proxy(d, kappa, kind="true") for d in dirs]
        sr_sur  = [safe_radius_proxy(d, kappa, kind="surrogate") for d in dirs]

        print("[DEMO] SU(3) commutator seed: surrogate vs true (Hessian + safe-radius proxy)")
        print(f"  At eps={eps:.3f}: min eig true={evals_true[0]:.6f}, surrogate={evals_sur[0]:.6f}")
        print(f"    true eig[0:6]={evals_true[:6]}")
        print(f"    sur  eig[0:6]={evals_sur[:6]}")
        print(f"  grid-safe eps (12 dirs): median true={np.median(sr_true):.3f}, surrogate={np.median(sr_sur):.3f}")
        print("  (Single-site proxy only; full lattice safe region will be smaller.)")
        print()


# ================================================================
# 4) CASRG MODEL2: EIGEN SOLVER SWAP (LOBPCG) + SCAN HARNESS
# ================================================================
def lowest_eigs_lobpcg(H, k=4, tol=1e-8, niter=250, seed=0):
    """
    H: scipy.sparse matrix or LinearOperator, symmetric.
    Returns sorted eigenvalues (lowest k).
    """
    if not SCIPY_OK:
        raise RuntimeError("SciPy not available.")
    n = H.shape[0]
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, k))
    # LOBPCG sometimes likes float64
    X = X.astype(np.float64, copy=False)
    evals, _ = spla.lobpcg(H, X, tol=tol, maxiter=niter, largest=False)
    evals = np.sort(np.array(evals, dtype=float))
    return evals

def lowest_eigs_eigsh(H, k=4, tol=1e-10, niter=500):
    if not SCIPY_OK:
        raise RuntimeError("SciPy not available.")
    evals = spla.eigsh(H, k=k, which="SA", tol=tol, maxiter=niter, return_eigenvectors=False)
    return np.sort(np.array(evals, dtype=float))

def lowest_eigs_shiftinvert(H, k=4, sigma=0.0, tol=1e-12, niter=500):
    if not SCIPY_OK:
        raise RuntimeError("SciPy not available.")
    # shift-invert: solves (H - sigma I)^{-1} v = mu v, eigenvalues relate by lambda = sigma + 1/mu
    evals = spla.eigsh(H, k=k, sigma=sigma, which="LM", tol=tol, maxiter=niter, return_eigenvectors=False)
    return np.sort(np.array(evals, dtype=float))

def solve_lowest_eigs(H, solver="lobpcg", k=4, **kw):
    if solver == "lobpcg":
        return lowest_eigs_lobpcg(H, k=k, **kw)
    if solver == "eigsh":
        return lowest_eigs_eigsh(H, k=k, **kw)
    if solver in ("shift-invert", "shiftinvert"):
        return lowest_eigs_shiftinvert(H, k=k, **kw)
    raise ValueError("solver must be one of: 'lobpcg', 'eigsh', 'shift-invert'")

def casrg_gap_scan(build_H_fn, eps_bits, truncations, Ls, solver="lobpcg", k=4, **solver_kw):
    """
    build_H_fn must be YOUR function:
        H = build_H_fn(L=<int>, trunc=<any>, eps_bit=<float>)
    returning a scipy sparse symmetric matrix (or LinearOperator) for the Hamiltonian.

    Returns list of dicts with evals + gap.
    """
    out = []
    for L in Ls:
        for trunc in truncations:
            for eps in eps_bits:
                H = build_H_fn(L=L, trunc=trunc, eps_bit=eps)
                evals = solve_lowest_eigs(H, solver=solver, k=k, **solver_kw)
                gap = float(evals[1] - evals[0]) if len(evals) >= 2 else float("nan")
                out.append({"L": L, "trunc": trunc, "eps_bit": eps, "evals": evals, "gap": gap})
                print(f"[CasRG] L={L} trunc={trunc} eps_bit={eps:g}  evals[0:4]={evals[:4]}  gap={gap:.6e}")
    return out


def demo_casrg_solver_patch():
    if not SCIPY_OK:
        return
    # Toy sparse SPD-ish operator with a controllable small gap
    n = 400
    diag = np.linspace(0.0, 10.0, n)
    diag[1] = 0.02  # enforce a small gap between ground and first excited-ish
    H = sp.diags(diag, offsets=0, format="csr")

    e_lob = solve_lowest_eigs(H, solver="lobpcg", k=4, tol=1e-10, niter=200)
    e_sh  = solve_lowest_eigs(H, solver="eigsh",  k=4, tol=1e-12, niter=500)

    print("[DEMO] CasRG patch: solver sanity on a toy sparse H")
    print("  lobpcg evals:", e_lob)
    print("  eigsh  evals:", e_sh)
    print("  (For clustered / ill-conditioned spectra, prefer eigsh or shift-invert.)")
    print()


# ================================================================
# RUN ALL DEMOS (safe defaults)
# ================================================================
RUN_DEMO_MASS = True
RUN_DEMO_SU2  = True
RUN_DEMO_SU3  = True
RUN_DEMO_CASRG_PATCH = True

if __name__ == "__main__":
    if RUN_DEMO_MASS:
        demo_mass_module_synthetic_4d_yukawa()

    if RUN_DEMO_SU2 and JAX_OK:
        demo_su2_rg_intertwining()

    if RUN_DEMO_SU3 and JAX_OK:
        demo_su3_comm_seed()

    if RUN_DEMO_CASRG_PATCH:
        demo_casrg_solver_patch()

    print("[DONE]")
    print(" - Mass module: measure_mass_yukawa_prefactor(G_1d, L, direction, ...) (tune rmin, W, tail_frac)")
    print(" - SU(2) RG: batch_R_both(...) uses keyword-safe static args and clamps qvec coords to avoid NaNs")
    print(" - SU(3): su3_comm_seed_true(...) replaces surrogate; demo compares Hessians + safe-radius proxy")
    print(" - CasRG: swap lanczos -> lowest_eigs_lobpcg(Hs,...) and scan with casrg_gap_scan(...)")


[DEMO] Mass module on synthetic 4D Yukawa tail
  true m=0.550000   measured m=0.550121  (MAD~1.75e-04, window (43.5, 50.5))

[WARN] NaNs detected in SU(2) RG demo. Reduce eps (try 0.05) and/or increase iters (try 8).
[DEMO] SU(2) RG intertwining: intrinsic (algebra) vs qvec-like gradients
  intrinsic (X) R: mean=nan median=nan max=nan
  qvec-like (Z) R: mean=nan median=nan max=nan
  Z/X median ratio ≈ nan   (expect ~4.0)
  X/Z median ratio ≈ nan   (expect ~0.25)
  expected intrinsic ~1/n = 0.062500, expected qvec-like ~4/n = 0.250000



KeyboardInterrupt: 